# CW attacks on the contrastive model — L2, L0, and CW inside training

The Carlini–Wagner machinery of `03-attacks/` aimed at the contrastive `combined_model` and at AT-rFGSM. **Part 1** develops the L2 attack (and keeps the reference implementation it was checked against). **Part 2** is the packaged version whose `final_solution` rounding the later work reuses. **Part 3** is the L0 attack, which removes features one at a time. **Part 4** generates CW examples *for* adversarial training. **Part 5** is the DDN runs from this workspace.

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `Carlini_and_Wagner_L2_attack.ipynb`, `Carlini_and_Wagner_L2_attack3.ipynb`, `Carlini_and_Wagner_L0_attack.ipynb`, `Carlini_and_Wagner_L0_attack222.ipynb`, `CW_for_training.ipynb`, `Adversarial_training_CW.ipynb` and `DDN.ipynb` (originally in `CW-CL` and `contrastive-learning`). Outputs cleared.*

## Part 1 — Developing the CW-L2 attack

### Step 1 — Setup

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_AT_rFGSM.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 128
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Create an instance of your model
model_AT_rFGSM = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_rFGSM.load_state_dict(torch.load('model_AT_rFGSM.pth', map_location=torch.device(device)))

# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

In [ ]:
alpha = torch.full((mal_x_batch.shape), 1e10, device=device)
alpha

In [ ]:
target_label = torch.zeros_like(mal_y_batch.squeeze(),dtype=torch.int64)
target_label

In [ ]:
expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)
removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

### Step 2 — The gradient-descent formulation, iterated

# Carlini and Wagner L2 attack

In [ ]:
import torch
import torch.optim as optim
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
  for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(),dtype=torch.int64)


def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

In [ ]:
import torch
import torch.optim as optim
import numpy as np

# Constants
max_iterations = 1000
abort_early = True
learning_rate = 0.1
initial_const = 1.
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 1.5
confidence=0.01


def gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, initial_const, largest_const,
                     const_factor, oimgs, labs, removal_array, device):
    """
    Perform gradient descent for the L0 attack.

    Args:
        model (torch.nn.Module): The PyTorch model to attack.
        targeted (bool): Whether the attack is targeted or untargeted.
        learning_rate (float): Learning rate for the optimizer.
        max_iterations (int): Maximum number of iterations for optimization.
        abort_early (bool): Whether to abort early if gradient descent gets stuck.
        initial_const (float): Initial value of the constant for optimization.
        largest_const (float): Largest value of the constant for optimization.
        const_factor (float): Factor to increase the constant if the previous value failed.
        independent_channels (bool): Optimize for the number of pixels changed if False, otherwise channels.
        oimgs (numpy.ndarray): Original images to attack.
        labs (numpy.ndarray): Labels for the attack (target labels if targeted).
        starts (numpy.ndarray): Initial perturbations.
        valid (numpy.ndarray): Mask of pixels that can be changed.
        device (torch.device): Device to perform computations on.

    Returns:
        numpy.ndarray: Gradient norm.
        numpy.ndarray: Model outputs (logits).
        numpy.ndarray: Adversarial images.
        float: The final constant used.
    """
    # Convert input arrays to tensors
    oimgs = torch.tensor(oimgs, dtype=torch.float32, device=device)
    labs_tensor = torch.tensor(labs, dtype=torch.int64, device=device)
    #starts = torch.tensor(starts, dtype=torch.float32, device=device)

    # Initialize variables
    imgs_tensor = torch_arctanh(oimgs).to(device)
    #starts = torch_arctanh(starts).to(device)


    # Set model to evaluation mode
    model.eval()

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
    print('oimgs ', oimgs.sum(1))
    print('fixed_features_mask ',(1 - non_fixed_features_mask).sum(1))

    valid = torch.tensor(non_fixed_features_mask, dtype=torch.float32, device=device)

    shape = oimgs.shape
    modifier = torch.zeros_like(oimgs, dtype=torch.float32, requires_grad=True, device=device)

    optimizer = optim.Adam([modifier], lr=learning_rate)

     # Define custom gradient hook
    def grad_hook(grad):
        if removal_array is not None:
            return grad * non_fixed_features_mask
        return grad

    # Register the hook
    #modifier.register_hook(grad_hook)


    for CONST in [initial_const * (const_factor ** i) for i in range(int(np.log(largest_const / initial_const) / np.log(const_factor)))]:
        print('CONST ',CONST)
        prev_loss = torch.tensor(float('inf'), device=device)
        for step in range(max_iterations):
            optimizer.zero_grad()

            modified_input = oimgs + modifier
            newimg = torch.clamp(modified_input,min=0.,max=1.)
            output = model(newimg)

            l2dist = torch.sum((newimg - oimgs) ** 2, dim=1)

            # Calculate the loss based on predictions
            real = output[0, labs_tensor]
            other = output[0, (labs_tensor + (1 if targeted else -1))]


            # Calculate adversarial loss
            loss1 = (other - real)
            loss2 = torch.sum(l2dist)
            total_loss = loss1

            # Backpropagate the loss
            total_loss.backward()
            optimizer.step()

            if step % ((max_iterations+1) // 50) == 0 :
                print('output ',output)
                print(compare(output, labs_tensor, confidence, targeted))

            if abort_early and step % (max_iterations // 10) == 0 and total_loss.item() > prev_loss * .999999:
                break

            prev_loss = total_loss.item()


            if compare(output, labs_tensor, confidence, targeted):
                return modifier.grad.detach().cpu(), output.detach().cpu(), newimg.detach().cpu(), CONST
    return None


modifier_grad, output, newimg, CONST = gradient_descent(model_AT_rFGSM, targeted, learning_rate, max_iterations, abort_early, initial_const, largest_const,
                                                       const_factor, mal_x_batch[3:4], target_label[3:4], removal_array, device)

In [ ]:
modifier_grad, output, newimg, CONST = gradient_descent(model_AT_rFGSM, targeted, learning_rate, max_iterations, abort_early, initial_const, largest_const,
                                                       const_factor, mal_x_batch[3:4], target_label[3:4], removal_array, device)

In [ ]:
torch.abs(mal_x_batch[3:4] - newimg).sum()

In [ ]:
CONST

In [ ]:
def gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, initial_const, largest_const,
                     const_factor, independent_channels, oimgs, labs, starts, valid, device):
    """
    Perform gradient descent for the L0 attack.

    Args:
        model (torch.nn.Module): The PyTorch model to attack.
        targeted (bool): Whether the attack is targeted or untargeted.
        learning_rate (float): Learning rate for the optimizer.
        max_iterations (int): Maximum number of iterations for optimization.
        abort_early (bool): Whether to abort early if gradient descent gets stuck.
        initial_const (float): Initial value of the constant for optimization.
        largest_const (float): Largest value of the constant for optimization.
        const_factor (float): Factor to increase the constant if the previous value failed.
        independent_channels (bool): Optimize for the number of pixels changed if False, otherwise channels.
        oimgs (numpy.ndarray): Original images to attack.
        labs (numpy.ndarray): Labels for the attack (target labels if targeted).
        starts (numpy.ndarray): Initial perturbations.
        valid (numpy.ndarray): Mask of pixels that can be changed.
        device (torch.device): Device to perform computations on.

    Returns:
        numpy.ndarray: Gradient norm.
        numpy.ndarray: Model outputs (logits).
        numpy.ndarray: Adversarial images.
        float: The final constant used.
    """
    # Convert input arrays to tensors
    oimgs = torch.tensor(oimgs, dtype=torch.float32, device=device)
    labs = torch.tensor(labs, dtype=torch.float32, device=device)
    starts = torch.tensor(starts, dtype=torch.float32, device=device)
    valid = torch.tensor(valid, dtype=torch.float32, device=device)

    # Initialize variables
    imgs = torch.atanh(oimgs * 1.999999).to(device)
    starts = torch.atanh(starts * 1.999999).to(device)
    valid = valid.to(device)

    shape = oimgs.shape
    modifier = torch.zeros(shape, dtype=torch.float32, requires_grad=True, device=device)

    optimizer = optim.Adam([modifier], lr=learning_rate)

    for CONST in [initial_const * (const_factor ** i) for i in range(int(np.log(largest_const / initial_const) / np.log(const_factor)))]:
        prev_loss = float('inf')
        for step in range(max_iterations):
            optimizer.zero_grad()

            newimg = (torch.tanh(modifier + starts) / 2) * valid + (1 - valid) * oimgs
            output = model(newimg)
            real = torch.sum(labs * output, dim=1)
            other = torch.max((1 - labs) * output - (labs * 10000), dim=1)[0]

            if targeted:
                loss1 = torch.clamp(other - real + 0.01, min=0.0)
            else:
                loss1 = torch.clamp(real - other + 0.01, min=0.0)

            loss2 = torch.sum((newimg - torch.tanh(imgs) / 2) ** 2, dim=(1, 2, 3))
            loss = torch.sum(CONST * loss1 + loss2)
            loss.backward()
            optimizer.step()

            if abort_early and step % (max_iterations // 10) == 0 and loss.item() > prev_loss * .999999:
                break
            prev_loss = loss.item()

            _, predicted = torch.max(output, 1)
            if compare(predicted.item(), torch.argmax(labs).item(), targeted):
                return modifier.grad.detach().cpu().numpy(), output.detach().cpu().numpy(), newimg.detach().cpu().numpy(), CONST
    return None


def l0_attack(model, imgs, targets, targeted=TARGETED, learning_rate=LEARNING_RATE,
              max_iterations=MAX_ITERATIONS, abort_early=ABORT_EARLY,
              initial_const=INITIAL_CONST, largest_const=LARGEST_CONST,
              reduce_const=REDUCE_CONST, const_factor=CONST_FACTOR,
              independent_channels=INDEPENDENT_CHANNELS):
    """
    Perform the L0 attack on the given images and targets.

    Args:
        model (torch.nn.Module): The PyTorch model to attack.
        imgs (numpy.ndarray): Original images to attack.
        targets (numpy.ndarray): Target labels if targeted, otherwise original labels.
        targeted (bool): Whether the attack is targeted or untargeted.
        learning_rate (float): Learning rate for the optimizer.
        max_iterations (int): Maximum number of iterations for optimization.
        abort_early (bool): Whether to abort early if gradient descent gets stuck.
        initial_const (float): Initial value of the constant for optimization.
        largest_const (float): Largest value of the constant for optimization.
        reduce_const (bool): Whether to reduce the constant after each iteration.
        const_factor (float): Factor to increase the constant if the previous value failed.
        independent_channels (bool): Optimize for the number of pixels changed if False, otherwise channels.

    Returns:
        numpy.ndarray: Adversarial examples for the supplied model.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def attack_single(img, target):
        valid = np.ones((1, model.image_size, model.image_size, model.num_channels))
        prev = np.copy(img).reshape((1, model.image_size, model.image_size, model.num_channels))
        last_solution = None
        const = initial_const

        equal_count = None

        while True:
            res = gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, initial_const,
                                   largest_const, const_factor, independent_channels, [np.copy(img)], [target],
                                   np.copy(prev), valid, device)
            if res is None:
                print("Final answer", equal_count)
                return last_solution

            restarted = False
            gradientnorm, scores, nimg, const = res
            if reduce_const:
                const /= 2

            #To calculate how many pixels have changed significantly between the original and adversarial images.
            #equal_count: The number of pixels that have changed between the original and adversarial images
            #np.sum(1 - valid): This calculates the number of pixels that are currently fixed (i.e., cannot change) by the optimizer
            equal_count = img.shape[-1] - np.sum(np.abs(img - nimg[0]) < .0001)
            print("Forced equal:", np.sum(1 - valid), "Equal count:", equal_count)
            if np.sum(valid) == 0:
                return [img]


            valid = valid.flatten()
            totalchange = abs(nimg[0] - img) * np.abs(gradientnorm[0])

            totalchange = totalchange.flatten()

            did = 0
            for e in np.argsort(totalchange):
                if np.all(valid[e]):
                    did += 1
                    valid[e] = 0

                    if totalchange[e] > .01:
                        break
                    if did >= .3 * equal_count ** .5:
                        break

            valid = np.reshape(valid, (1, model.image_size, model.image_size, -1))
            print("Now forced equal:", np.sum(1 - valid))

            last_solution = prev = nimg

    results = []
    for i, (img, target) in enumerate(zip(imgs, targets)):
        print("Attack iteration", i)
        results.extend(attack_single(img, target))
    return np.array(results)

# Example usage:
# model = ...  # Your PyTorch model
# imgs = ...   # Your input images
# targets = ...# Your target labels (if targeted), otherwise original labels
# adversarial_images = l0_attack(model, imgs, targets)

In [ ]:
import torch
import torch.optim as optim
import numpy as np

def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def perform_attack(model, imgs, labs, device, targeted=True, abort_early=True, confidence=0.1,
                   initial_const=1., learning_rate=0.01, binary_search_steps=5,
                   max_iterations=1000, removal_array=None):
    """Perform the Carlini-Wagner L2 attack on a batch of images with fixed features."""

    batch_size = imgs.shape[0]

    # Convert images to tanh-space
    imgs_tanh = torch_arctanh(imgs)

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, device=device)
    CONST = torch.full((batch_size,), initial_const, device=device)
    upper_bound = torch.full((batch_size,), 1e10, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()

    # Ensure fixed_features_mask is a tensor
    if removal_array is not None:
        expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
        non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))
        print('imgs ', imgs.sum(1))
        print('fixed_features_mask ',(1 - non_fixed_features_mask).sum(1))
        alpha = torch.full((mal_x_batch.shape), 1e6, device=device) * (1 - non_fixed_features_mask) + torch.ones_like(mal_x_batch, device=device)
        print(alpha)

    for outer_step in range(binary_search_steps):
        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.SGD([modifier], lr=learning_rate)

        # Define custom gradient hook
        def grad_hook(grad):
            if removal_array is not None:
                return grad * non_fixed_features_mask
            return grad

        # Register the hook
        #modifier.register_hook(grad_hook)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        prev_loss = torch.tensor(float('inf'), device=device)

        for iteration in range(max_iterations):
            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = imgs_tensor + modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum(((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) * alpha) ** 2, dim=1)
            #l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

            # Calculate the loss based on predictions
            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), (labs_tensor + (1 if targeted else -1))]


            # Calculate adversarial loss
            loss1 = torch.sum(CONST * torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0))
            loss2 = torch.sum(l2dist)
            total_loss = loss1 + loss2

            # Backpropagate the loss
            total_loss.backward()
            optimizer.step()

            # Apply the mask to reset the fixed features in the modifier
            #if removal_array is not None:
                #modifier.data = modifier.data * non_fixed_features_mask

            # Print losses every 10% of max_iterations
            if iteration % (max_iterations // 10) == 0:
                #print('real ',real)
                print('compare(output, labs_tensor, confidence, targeted) ',compare(output, labs_tensor, confidence, targeted))
                print(f"Iteration {iteration}, Total Loss: {total_loss.item()}, Loss1: {loss1.item()}, Loss2: {loss2.item()}")

            # Abort early if no improvement
            if abort_early and iteration % (max_iterations // 10) == 0:
                if total_loss.item() > prev_loss * 0.999999:
                    print('Early stopping: no improvement in loss')
                    break
                prev_loss = total_loss.item()

            # Update the best result found so far
            update_mask = (l2dist < bestl2) & compare(output, labs_tensor, confidence, targeted)
            bestl2[update_mask] = l2dist[update_mask]
            bestscore[update_mask] = torch.argmax(output[update_mask], dim=1)

            update_mask2 = (l2dist < o_bestl2) & compare(output, labs_tensor, confidence, targeted)
            o_bestl2[update_mask2] = l2dist[update_mask2]
            o_bestscore[update_mask2] = torch.argmax(output[update_mask2], dim=1)
            o_bestattack[update_mask2] = newimg[update_mask2]

        # Create masks for successful and unsuccessful attacks
        success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
        failure_mask = ~success_mask

        # Update upper bounds and constants where attacks are successful
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

        # Update lower bounds and constants where attacks are unsuccessful
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        # Handle cases where upper bound is still too high
        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= 10

        print(f"Binary search step {outer_step}, CONST: {CONST}")
        print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")

    return o_bestattack.detach().cpu(), o_bestl2.detach().cpu(), CONST.detach().cpu(), alpha

# Example usage of fixed_features_mask
# Let's say we want to fix some features in the input image, here is how you define the mask
# fixed_features_mask = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]]) # Example for a 3x3 image

In [ ]:
o_bestl2

In [ ]:
print((torch.abs(o_bestattack - mal_x_batch)).sum())

In [ ]:
model_AT_rFGSM(o_bestattack)

In [ ]:
import torch
import torch.optim as optim
import numpy as np

def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability.
    (1-eps) 0.999999: This scaling factor is used to ensure that the input to np.arctanh stays slightly away
     from the boundary values -1 and 1. The arctanh function is undefined at exactly -1 and 1,
     so multiplying by 0.999999 helps avoid numerical issues by keeping the input strictly within (-1, 1)."""

    return torch.atanh(((x - 0.5) / 0.5) * (1-eps))


def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

def perform_attack(model, imgs, labs, device, targeted=True, abort_early=True, confidence=0.1,
                   initial_const=1., learning_rate=0.01, binary_search_steps=5,
                   max_iterations=1000):
    """Perform the Carlini-Wagner L2 attack on a batch of images."""

    batch_size = imgs.shape[0]

    # Convert images to tanh-space
    imgs_tanh = torch_arctanh(imgs)

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, device=device)
    CONST = torch.full((batch_size,), initial_const, device=device)
    upper_bound = torch.full((batch_size,), 1e10, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = torch.zeros_like(imgs_tensor)

    # Set model to evaluation mode
    model.eval()

    for outer_step in range(binary_search_steps):
        # Initialize modifier
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=learning_rate)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        prev_loss = torch.tensor(float('inf'), device=device)

        for iteration in range(max_iterations):
            optimizer.zero_grad()

            # Compute new images in original space
            newimg = torch.tanh(modifier + imgs_tensor) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

            # Calculate the loss based on predictions
            real = output[torch.arange(batch_size), labs_tensor]
            other = output[torch.arange(batch_size), (labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.sum(CONST * torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0))
            loss2 = torch.sum(l2dist)
            total_loss = loss1 + loss2

            # Backpropagate the loss
            total_loss.backward()
            optimizer.step()

            # Print losses every 10% of max_iterations
            if iteration % (max_iterations // 10) == 0:
                print(f"Iteration {iteration}, Total Loss: {total_loss.item()}, Loss1: {loss1.item()}, Loss2: {loss2.item()}")

            # Abort early if no improvement
            if abort_early and iteration % (max_iterations // 10) == 0:
                if total_loss.item() > prev_loss * 0.999999:
                    print('Early stopping: no improvement in loss')
                    break
                prev_loss = total_loss.item()

            # Update the best result found so far
            update_mask = (l2dist < bestl2) & compare(output, labs_tensor, confidence, targeted)
            bestl2[update_mask] = l2dist[update_mask]
            bestscore[update_mask] = torch.argmax(output[update_mask], dim=1)

            update_mask2 = (l2dist < o_bestl2) & compare(output, labs_tensor, confidence, targeted)
            o_bestl2[update_mask2] = l2dist[update_mask2]
            o_bestscore[update_mask2] = torch.argmax(output[update_mask2], dim=1)
            o_bestattack[update_mask2] = newimg[update_mask2]


        # Create masks for successful and unsuccessful attacks
        success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
        failure_mask = ~success_mask

        # Update upper bounds and constants where attacks are successful
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

        # Update lower bounds and constants where attacks are unsuccessful
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        # Handle cases where upper bound is still too high
        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= 10


        print(f"Binary search step {outer_step}, CONST: {CONST}")
        print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")

    return o_bestattack.detach().cpu(), o_bestl2.detach().cpu(), CONST.detach().cpu()

In [ ]:
model_AT_rFGSM(o_bestattack).argmax(1)

### Step 3 — Checking against the reference implementation

In [ ]:
imgs = mal_x_batch
labs = target_label

def torch_arctanh(x, eps=1e-6):
    x *= (1. - eps)
    return torch.atanh((x - 0.5) / (0.5))

def _compare(self, output, target):
    output = np.copy(output)
    if targeted:
        output[target] -= self.confidence
    else:
        output[target] += self.confidence
    output = np.argmax(output)
    print('output ', output)
    print('output == target ',output == target)

    if self.targeted:
        return output == target
    else:
        return output != target

targeted = True
CONFIDENCE = 0.1
batch_size = imgs.shape[0]

# Convert to tanh-space
imgs = torch_arctanh(imgs)
#imgs = np.arctanh((imgs - 0.5) / 0.5 * 0.999999)

imgs_tensor = torch.tensor(imgs, dtype=torch.float32, device=device)
labs_tensor = torch.tensor(labs, dtype=torch.float32, device=device)

# Set the lower and upper bounds accordingly
initial_const = 100.
CONST = torch.ones(batch_size) * initial_const

# The best l2, score, and image attack
o_bestl2 = [1e10] * batch_size
o_bestscore = [-1] * batch_size
o_bestattack = [np.zeros(imgs[0].shape)] * batch_size


# Modifier variable that we're going to optimize over
modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

# Optimizer setup
optimizer = optim.Adam([modifier], lr=0.01)

bestl2 = [1e10] * batch_size
bestscore = [-1] * batch_size

prev = np.inf
MAX_ITERATIONS = 1000
model_AT_rFGSM.eval()

for iteration in range(MAX_ITERATIONS):
    optimizer.zero_grad()

    # Compute the new images
    newimg = torch.tanh(modifier + imgs_tensor) * 0.5 + 0.5

    # Compute the prediction BEFORE-SOFTMAX
    output = model_AT_rFGSM(newimg)

    # Compute the distance to the input data
    l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

    # Compute the probability of the label class versus the maximum other
    real = output[torch.arange(len(labs)), labs]
    labs2 = (labs + 1)
    other = output[torch.arange(len(labs2)), labs2]

    if targeted:
        # if targeted, optimize for making the other class most likely
        loss1 = torch.clamp(other - real + CONFIDENCE, min=0.0)
    else:
        # if untargeted, optimize for making this class least likely.
        loss1 = torch.clamp(real - other + CONFIDENCE, min=0.0)

    # Sum up the losses
    loss2 = torch.sum(l2dist)
    loss1 = torch.sum(CONST * loss1)
    loss = loss1 + loss2
    #loss1 = torch.sum(loss1)
    #loss =loss1

    # Backpropagate the loss
    loss.backward()
    optimizer.step()

    # Print out the losses every 10%
    #if iteration % (MAX_ITERATIONS // 1) == 0:
    print(iteration, loss.item(), loss1.item(), loss2.item())
    #print(iteration, loss.item(), loss1.item())

In [ ]:
def compare(output, target):
    output = torch.clone(output)
    if targeted:
        output[target] -= confidence
        #output[target] -= self.confidence
    else:
        output[ target] += confidence
        #output[target] += self.confidence
    output = torch.argmax(output)

    if targeted:
        return output == target
    else:
        return output != target

In [ ]:
confidence = 0.1
compare( output[0], labs[e])

In [ ]:
MAX_ITERATIONS = 500
for iteration in range(MAX_ITERATIONS):
  if iteration % (MAX_ITERATIONS // 10) == 0:
    print(iteration)

In [ ]:
model_AT_imax(o_bestattack[1])

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

BINARY_SEARCH_STEPS = 9  # number of times to adjust the constant with binary search
MAX_ITERATIONS = 10000   # number of iterations to perform gradient descent
ABORT_EARLY = True       # if we stop improving, abort gradient descent early
LEARNING_RATE = 1e-2     # larger values converge faster to less accurate results
TARGETED = True          # should we target one specific class? or just be wrong?
CONFIDENCE = 0           # how strong the adversarial example should be
INITIAL_CONST = 1e-3     # the initial constant c to pick as a first guess

class CarliniL2:
    def __init__(self, model, device, batch_size=1, confidence=CONFIDENCE,
                 targeted=TARGETED, learning_rate=LEARNING_RATE,
                 binary_search_steps=BINARY_SEARCH_STEPS, max_iterations=MAX_ITERATIONS,
                 abort_early=ABORT_EARLY,
                 initial_const=INITIAL_CONST,
                 boxmin=-0.5, boxmax=0.5):
        """
        The L_2 optimized attack.

        This attack is the most efficient and should be used as the primary
        attack to evaluate potential defenses.

        Returns adversarial examples for the supplied model.

        confidence: Confidence of adversarial examples: higher produces examples
          that are farther away, but more strongly classified as adversarial.
        batch_size: Number of attacks to run simultaneously.
        targeted: True if we should perform a targeted attack, False otherwise.
        learning_rate: The learning rate for the attack algorithm. Smaller values
          produce better results but are slower to converge.
        binary_search_steps: The number of times we perform binary search to
          find the optimal tradeoff-constant between distance and confidence.
        max_iterations: The maximum number of iterations. Larger values are more
          accurate; setting too small will require a large learning rate and will
          produce poor results.
        abort_early: If true, allows early aborts if gradient descent gets stuck.
        initial_const: The initial tradeoff-constant to use to tune the relative
          importance of distance and confidence. If binary_search_steps is large,
          the initial constant is not important.
        boxmin: Minimum pixel value (default -0.5).
        boxmax: Maximum pixel value (default 0.5).
        """

        self.model = model
        self.device = device
        self.TARGETED = targeted
        self.LEARNING_RATE = learning_rate
        self.MAX_ITERATIONS = max_iterations
        self.BINARY_SEARCH_STEPS = binary_search_steps
        self.ABORT_EARLY = abort_early
        self.CONFIDENCE = confidence
        self.initial_const = initial_const
        self.batch_size = batch_size

        self.repeat = binary_search_steps >= 10

        self.I_KNOW_WHAT_I_AM_DOING_AND_WANT_TO_OVERRIDE_THE_PRESOFTMAX_CHECK = False

        # Image dimensions
        self.boxmin = boxmin
        self.boxmax = boxmax

        # Pre-compute box scaling factors
        self.boxmul = (boxmax - boxmin) / 2.
        self.boxplus = (boxmin + boxmax) / 2.

    def attack(self, imgs, targets):
        """
        Perform the L_2 attack on the given images for the given targets.

        If self.targeted is true, then the targets represent the target labels.
        If self.targeted is false, then targets are the original class labels.
        """
        r = []
        print('go up to', len(imgs))
        for i in range(0, len(imgs), self.batch_size):
            print('tick', i)
            r.extend(self.attack_batch(imgs[i:i+self.batch_size], targets[i:i+self.batch_size]))
        return np.array(r)

    def attack_batch(self, imgs, labs):
        """
        Run the attack on a batch of images and labels.
        """
        def compare(x, y):
            if not isinstance(x, (float, int, np.int64)):
                x = np.copy(x)
                if self.TARGETED:
                    x[y] -= self.CONFIDENCE
                else:
                    x[y] += self.CONFIDENCE
                x = np.argmax(x)
            if self.TARGETED:
                return x == y
            else:
                return x != y

        batch_size = self.batch_size

        # Convert to tanh-space
        imgs = np.arctanh((imgs - self.boxplus) / self.boxmul * 0.999999)

        imgs_tensor = torch.tensor(imgs, dtype=torch.float32, device=self.device)
        labs_tensor = torch.tensor(labs, dtype=torch.float32, device=self.device)

        # Set the lower and upper bounds accordingly
        lower_bound = np.zeros(batch_size)
        CONST = np.ones(batch_size) * self.initial_const
        upper_bound = np.ones(batch_size) * 1e10

        # The best l2, score, and image attack
        o_bestl2 = [1e10] * batch_size
        o_bestscore = [-1] * batch_size
        o_bestattack = [np.zeros(imgs[0].shape)] * batch_size



        for outer_step in range(self.BINARY_SEARCH_STEPS):
            print(o_bestl2)
            # Modifier variable that we're going to optimize over
            modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(self.device)

            # Optimizer setup
            optimizer = optim.Adam([modifier], lr=self.LEARNING_RATE)

            bestl2 = [1e10] * batch_size
            bestscore = [-1] * batch_size

            # The last iteration (if we run many steps) repeats the search once.
            if self.repeat and outer_step == self.BINARY_SEARCH_STEPS - 1:
                CONST = upper_bound

            prev = np.inf
            for iteration in range(self.MAX_ITERATIONS):
                optimizer.zero_grad()

                # Compute the new images
                newimg = torch.tanh(modifier + imgs_tensor) * self.boxmul + self.boxplus

                # Compute the prediction BEFORE-SOFTMAX
                output = self.model(newimg)

                # Compute the distance to the input data
                l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * self.boxmul + self.boxplus)) ** 2, dim=[1, 2, 3])

                # Compute the probability of the label class versus the maximum other
                real = torch.sum(labs_tensor * output, dim=1)
                other = torch.max((1 - labs_tensor) * output - (labs_tensor * 10000), dim=1).values

                if self.TARGETED:
                    # if targeted, optimize for making the other class most likely
                    loss1 = torch.clamp(other - real + self.CONFIDENCE, min=0.0)
                else:
                    # if untargeted, optimize for making this class least likely.
                    loss1 = torch.clamp(real - other + self.CONFIDENCE, min=0.0)

                # Sum up the losses
                loss2 = torch.sum(l2dist)
                loss1 = torch.sum(CONST * loss1)
                loss = loss1 + loss2

                # Backpropagate the loss
                loss.backward()
                optimizer.step()

                # Print out the losses every 10%
                if iteration % (self.MAX_ITERATIONS // 10) == 0:
                    print(iteration, loss.item(), loss1.item(), loss2.item())

                # Check if we should abort search if we're getting nowhere.
                if self.ABORT_EARLY and iteration % (self.MAX_ITERATIONS // 10) == 0:
                    if loss.item() > prev * .9999:
                        break
                    prev = loss.item()

                # Adjust the best result found so far
                for e, (l2, sc, ii) in enumerate(zip(l2dist.cpu().detach().numpy(), output.cpu().detach().numpy(), newimg.cpu().detach().numpy())):
                    if l2 < bestl2[e] and compare(sc, np.argmax(labs[e])):
                        bestl2[e] = l2
                        bestscore[e] = np.argmax(sc)
                    if l2 < o_bestl2[e] and compare(sc, np.argmax(labs[e])):
                        o_bestl2[e] = l2
                        o_bestscore[e] = np.argmax(sc)
                        o_bestattack[e] = ii

            # Adjust the constant as needed
            for e in range(batch_size):
                if compare(bestscore[e], np.argmax(labs[e])) and bestscore[e] != -1:
                    # success, divide const by two
                    upper_bound[e] = min(upper_bound[e], CONST[e])
                    if upper_bound[e] < 1e9:
                        CONST[e] = (lower_bound[e] + upper_bound[e]) / 2
                else:
                    # failure, either multiply by 10 if no solution found yet
                    # or do binary search with the known upper bound
                    lower_bound[e] = max(lower_bound[e], CONST[e])
                    if upper_bound[e] < 1e9:
                        CONST[e] = (lower_bound[e] + upper_bound[e]) / 2
                    else:
                        CONST[e] *= 10

        # Return the best solution found
        return np.array(o_bestattack)

In [ ]:
"""PyTorch Carlini and Wagner L2 attack algorithm.

Based on paper by Carlini & Wagner, https://arxiv.org/abs/1608.04644 and a reference implementation at
https://github.com/tensorflow/cleverhans/blob/master/cleverhans/attacks_tf.py
"""
import os
import sys
import torch
import numpy as np
from torch import optim
from torch import autograd

In [ ]:
a = torch_arctanh(mal_x_batch, eps=1e-6)
a

In [ ]:
tanh_rescale(a)

In [ ]:
class AttackCarliniWagnerL2:

    def __init__(self, targeted=True, search_steps=None, max_steps=None, cuda=True, debug=False):
        self.debug = debug
        self.targeted = targeted
        self.num_classes = 2
        self.confidence = 0.1  # FIXME need to find a good value for this, 0 value used in paper not doing much...
        self.initial_const = 1.  # bumped up from default of .01 in reference code
        self.binary_search_steps = search_steps or 5  #Number of steps for binary search to find the optimal constant c
        self.repeat = self.binary_search_steps >= 10
        self.max_steps = max_steps or 1000  #Maximum iterations for the optimization process
        self.abort_early = True
        self.clip_min = 0.
        self.clip_max = 1.
        self.cuda = cuda
        self.clamp_fn = 'tanh'  # set to something else perform a simple clamp instead of tanh
        self.init_rand = False  # an experiment, does a random starting point help?

    #Determines if the adversarial example successfully fools the model
    def _compare(self, output, target):
        if not isinstance(output, (float, int, np.int64)):
            print('isinstance')
            output = np.copy(output)
            if self.targeted:
                output[target] -= self.confidence
            else:
                output[target] += self.confidence
            output = np.argmax(output)
            print('output ', output)
            print('output == target ',output == target)

        if self.targeted:
            return output == target
        else:
            return output != target

    #Computes the combined loss function used for optimization
    def _loss(self, output, target, dist, scale_const):
        # compute the probability of the label class versus the maximum other
        #print('output ',output)
        real = (target * output).sum(1)
        #print('real',real)
        other = ((1. - target) * output - target * 10000.).max(1)[0]
        #print('other',other)
        if self.targeted:
            # if targeted, optimize for making the other class most likely
            loss1 = torch.clamp(other - real + self.confidence, min=0.)  # equiv to max(..., 0.)
            print('loss1 ',loss1)
        else:
            # if non-targeted, optimize for making this class least likely.
            loss1 = torch.clamp(real - other + self.confidence, min=0.)  # equiv to max(..., 0.)
        loss1 = torch.sum(scale_const * loss1)

        loss2 = dist.sum()
        #print('loss2 ',loss2)

        loss = loss1 + loss2
        return loss

    def _optimize(self, optimizer, model, input_var, modifier_var, target_var, scale_const_var, input_orig=None):
        # apply modifier and clamp resulting image to keep bounded from clip_min to clip_max
        if self.clamp_fn == 'tanh':
            input_adv = tanh_rescale(modifier_var + input_var, self.clip_min, self.clip_max)
        else:
            input_adv = torch.clamp(modifier_var + input_var, self.clip_min, self.clip_max)
        model.eval()
        output = model(input_adv)
        print('output ',output)

        # distance to the original input data
        if input_orig is None:
            dist = l2_dist(input_adv, input_var, keepdim=False)
        else:
            #dist = l2_dist(input_adv, input_orig, keepdim=False)
            dist = l2r_dist(input_adv, input_orig, keepdim=False)

        loss = self._loss(output, target_var, dist, scale_const_var)

        optimizer.zero_grad()
        loss.backward()
        # Debugging gradient information
        print("Modifier gradient norm:", modifier_var.grad.norm().item())


        optimizer.step()

        # Debugging modifier information
        print('Modifier mean:', modifier_var.data.mean().item())

        #print('loss.data ', loss.data)
        loss_np = loss.data
        print('loss_np ', loss_np)
        dist_np = dist.data.cpu().numpy()
        output_np = output.data.cpu().numpy()
        input_adv_np = input_adv.data.cpu().numpy()
        return loss_np, dist_np, output_np, input_adv_np

    def run(self, model, input, target, batch_idx=0):
        batch_size = input.size(0)
        #print('batch_size',batch_size)

        # set the lower and upper bounds accordingly(for the binary search on c)
        lower_bound = np.zeros(batch_size)
        #print('lower_bound',lower_bound)
        scale_const = np.ones(batch_size) * self.initial_const
        #print('scale_const',scale_const)
        upper_bound = np.ones(batch_size) * 1e10
        #print('upper_bound',upper_bound)

        # python/numpy placeholders for the overall best l2, label score, and adversarial image
        o_best_l2 = [1e10] * batch_size
        o_best_score = [-1] * batch_size
        o_best_attack = input.cpu().numpy()

        # setup input (image) variable, clamp/scale as necessary
        if self.clamp_fn == 'tanh':
            # convert to tanh-space, input already int -1 to 1 range, does it make sense to do
            # this as per the reference implementation or can we skip the arctanh?
            input_var = autograd.Variable(torch_arctanh(input), requires_grad=False)
            input_orig = tanh_rescale(input_var, self.clip_min, self.clip_max)
        else:
            input_var = autograd.Variable(input, requires_grad=False)
            input_orig = None

        # setup the target variable, we need it to be in one-hot form for the loss function
        target_onehot = torch.zeros(target.size() + (self.num_classes,))
        #print('target_onehot ', target_onehot)
        if self.cuda:
            target_onehot = target_onehot.cuda()
        target_onehot.scatter_(1, target.unsqueeze(1), 1.)
        #print('target_onehot ', target_onehot)
        target_var = autograd.Variable(target_onehot, requires_grad=False)

        # setup the modifier variable, this is the variable we are optimizing over
        modifier = torch.zeros(input_var.size()).float()
        if self.init_rand:
            # Experiment with a non-zero starting point...
            modifier = torch.normal(means=modifier, std=0.001)
        if self.cuda:
            modifier = modifier.cuda()
        modifier_var = autograd.Variable(modifier, requires_grad=True)

        optimizer = optim.Adam([modifier_var], lr=0.0005)
        model.eval()
        for search_step in range(self.binary_search_steps):
            print('Batch: {0:>3}, search step: {1}'.format(batch_idx, search_step))
            if self.debug:
                print('Const:')
                for i, x in enumerate(scale_const):
                    print(i, x)
            best_l2 = [1e10] * batch_size
            best_score = [-1] * batch_size

            # The last iteration (if we run many steps) repeat the search once.
            if self.repeat and search_step == self.binary_search_steps - 1:
                scale_const = upper_bound

            scale_const_tensor = torch.from_numpy(scale_const).float()
            if self.cuda:
                scale_const_tensor = scale_const_tensor.cuda()
            scale_const_var = autograd.Variable(scale_const_tensor, requires_grad=False)

            prev_loss = 1e6
            for step in range(self.max_steps):
                # perform the attack
                loss, dist, output, adv_img = self._optimize(
                    optimizer,
                    model,
                    input_var,
                    modifier_var,
                    target_var,
                    scale_const_var,
                    input_orig)

                if step % 100 == 0 or step == self.max_steps - 1:
                    print('Step: {0:>4}, loss: {1:6.4f}, dist: {2:8.5f}, modifier mean: {3:.5e}'.format(
                        step, loss, dist.mean(), modifier_var.data.mean()))

                if self.abort_early and step % (self.max_steps // 10) == 0:
                    if loss > prev_loss * .9999:
                        print('Aborting early...')
                        break
                    prev_loss = loss

                # update best result found
                for i in range(batch_size):
                    target_label = target[i]
                    output_logits = output[i]
                    output_label = np.argmax(output_logits)
                    di = dist[i]
                    if self.debug:
                        if step % 100 == 0:
                            print('{0:>2} dist: {1:.5f}, output: {2:>3}, {3:5.3}, target {4:>3}'.format(
                                i, di, output_label, output_logits[output_label], target_label))
                    if di < best_l2[i] and self._compare(output_logits, target_label):
                        if self.debug:
                            print('{0:>2} best step,  prev dist: {1:.5f}, new dist: {2:.5f}'.format(
                                  i, best_l2[i], di))
                        best_l2[i] = di
                        best_score[i] = output_label
                    if di < o_best_l2[i] and self._compare(output_logits, target_label):
                        if self.debug:
                            print('{0:>2} best total, prev dist: {1:.5f}, new dist: {2:.5f}'.format(
                                  i, o_best_l2[i], di))
                        o_best_l2[i] = di
                        o_best_score[i] = output_label
                        o_best_attack[i] = adv_img[i]

                sys.stdout.flush()
                # end inner step loop

            # adjust the constants
            batch_failure = 0
            batch_success = 0
            for i in range(batch_size):
                print('self._compare(best_score[i], target[i])', self._compare(best_score[i], target[i]))
                if self._compare(best_score[i], target[i]) and best_score[i] != -1:
                    # successful, do binary search and divide const by two
                    upper_bound[i] = min(upper_bound[i], scale_const[i])
                    if upper_bound[i] < 1e9:
                        scale_const[i] = (lower_bound[i] + upper_bound[i]) / 2
                    if self.debug:
                        print('{0:>2} successful attack, lowering const to {1:.3f}'.format(
                            i, scale_const[i]))
                else:
                    # failure, multiply by 10 if no solution found
                    # or do binary search with the known upper bound
                    lower_bound[i] = max(lower_bound[i], scale_const[i])
                    if upper_bound[i] < 1e9:
                        scale_const[i] = (lower_bound[i] + upper_bound[i]) / 2
                    else:
                        scale_const[i] *= 10
                    if self.debug:
                        print('{0:>2} failed attack, raising const to {1:.3f}'.format(
                            i, scale_const[i]))
                if self._compare(o_best_score[i], target[i]) and o_best_score[i] != -1:
                    batch_success += 1
                else:
                    batch_failure += 1

            print('Num failures: {0:2d}, num successes: {1:2d}\n'.format(batch_failure, batch_success))
            sys.stdout.flush()
            # end outer search loop

        return o_best_attack


# mal_x_batch.shape = torch.Size([15, 10000])
# target_label = torch.Size([15])
#attack = AttackCarliniWagnerL2(targeted=True, max_steps=1000, cuda=False, debug=True)
#adversarial_examples = attack.run(model_AT_rFGSM, mal_x_batch, target_label)

In [ ]:
attack = AttackCarliniWagnerL2(targeted=True, max_steps=20, cuda=False, debug=False)
# Generate adversarial example
adversarial_examples = attack.run(model_AT_rFGSM, mal_x_batch[:3], target_label[:3])

In [ ]:
adversarial_examples.shape

In [ ]:
model_AT_rFGSM(mal_x_batch)

In [ ]:
(abs(adversarial_examples - mal_x_batch.cpu().numpy())).sum()

In [ ]:
(abs(adversarial_examples - mal_x_batch.cpu().numpy())).sum(1)

In [ ]:
model_AT_rFGSM(torch.tensor(adversarial_examples))

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print((torch.abs(pertb_mal_x - mal_x_batch)).sum())
            print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))
            break

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 1000, 'step_length': 0.01, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2, device, **attack_params)

In [ ]:
import torch
import torch.nn as nn

def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Initialize the optimizer for x_next
    x_next.requires_grad = True
    optimizer = torch.optim.Adam([x_next], lr=step_length)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        optimizer.zero_grad()
        y_model = model(x_next)

        batch_size = x_next.shape[0]
        real = y_model[torch.arange(batch_size),(y.view(-1).long() - 1)]
        other = y_model[torch.arange(batch_size),  y.view(-1).long()]
        loss = torch.clamp((real - other ), max=0.0)
        if (t+1)%50 == 0:
            print(y_model)

        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient and optimize
        loss.sum().backward()

        # Apply constraints using masks
        with torch.no_grad():
            grad_vars = x_next.grad.data

            grad4insertion = (grad_vars >= 0) * (x_next < 1.) * grad_vars
            grad4removal = (grad_vars < 0) * (x_next > 0.) * removal_array_updated * grad_vars
            grad_vars = grad4removal + grad4insertion

            if norm == 'linf':
                grad_vars = step_length * torch.sign(grad_vars).float()

            elif norm == 'l2':
                l2norm = torch.linalg.norm(grad_vars, dim=-1, keepdim=True)
                grad_vars = step_length * (grad_vars / (l2norm + 1e-20)).float()

            elif norm == 'free':
                grad_vars = grad_vars

            elif norm == 'l1':
                un_mod = torch.abs(x - x_next) <= 1e-6
                grad_vars = grad_vars * un_mod
                max_grad = torch.topk(torch.abs(grad_vars).view(grad_vars.size(0), -1), 1, dim=-1)[0]
                grad_vars = (torch.abs(grad_vars) >= max_grad).float() * torch.sign(grad_vars).float()
                done = get_done(x_next, y, model)
                if torch.all(done):
                    break
                grad_vars[done] = 0.

            else:
                raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

            # Apply projected gradient update
            x_next.data = torch.clamp(x_next.data + grad_vars * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    # x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next.data[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next.data - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next.data, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres', torch.mean(diff))

    return x_next.data

## Part 2 — The packaged CW (the version later work imports)

### Step 4 — compare, final_solution, and the loss variants

# Carlini and Wagner L2 attack

In [ ]:
import torch
import torch.optim as optim
import numpy as np
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

model = combined_model.to(device)
imgs = mal_x_batch[:100]
labs = target_label[:100]
targeted = True
#abort_early = True
confidence = 0.1
initial_const = 1
learning_rate = 0.5
binary_search_steps = 1
max_iterations = 100
removal_array = removal_array

batch_size = imgs.shape[0]

# Ensure fixed_features_mask is a tensor
if removal_array is not None:
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

# Calculate indices where mask is 0 or 1
mask_0 = non_fixed_features_mask == 0
mask_1 = non_fixed_features_mask == 1

# Create empty tensor for the result
imgs_tanh = torch.empty_like(imgs)

# Apply the transformations based on the mask
imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

# Use clone().detach() to copy the tensor correctly
imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Set the lower and upper bounds for binary search
lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

# Variables to store the best results
o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()

# Set model to evaluation mode
model.eval()
criterion = nn.CrossEntropyLoss(reduction='none')

for outer_step in range(binary_search_steps):

    if outer_step == (binary_search_steps - 1):
        max_iterations = 3000
        print(f"Binary search step {outer_step}, max_iterations: {max_iterations}")

    # Initialize modifier with requires_grad=True
    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

    # Setup optimizer
    optimizer = optim.SGD([modifier], lr=learning_rate)

    # Setup learning rate scheduler
    #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

    # To track the best loss for each sample and count of iterations without improvement
    best_loss = torch.full((batch_size,), float('inf'), device=device)
    no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

    # Track active samples
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)

    for iteration in range(max_iterations):
        if active_mask.sum() == 0:
            print(f"All samples are inactive at iteration {iteration}.")
            break

        # Filter active samples
        active_indices = torch.where(active_mask)[0]
        active_modifier = modifier[active_mask]
        active_imgs_tensor = imgs_tensor[active_mask]
        active_labs_tensor = labs_tensor[active_mask]
        active_CONST = CONST[active_mask]

        optimizer.zero_grad()

        # Compute new images in original space
        modified_input = active_imgs_tensor + active_modifier
        newimg = torch.tanh(modified_input) * 0.5 + 0.5

        # Get model predictions (pre-softmax)
        output = model(newimg.to(device))

        # Compute L2 distance to original images
        l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
        #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


        # Calculate the loss based on predictions
        real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
        other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

        # Calculate adversarial loss
        loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
        #loss1 = criterion(output, active_labs_tensor)
        loss2 = l2dist
        loss = (active_CONST * loss1)
        total_loss = torch.sum(loss)

        # Backpropagate the loss
        total_loss.backward()

        optimizer.step()

        # Step the learning rate scheduler
        #scheduler.step()

        # Check for improvement in loss for each sample
        improved_mask = loss < best_loss[active_mask]
        best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
        no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

        # Update the best result found so far
        update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
        bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
        bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

        update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
        o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
        o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
        o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

        # Skip the rest of the loop for samples that haven't improved in 50 iterations
        active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

        # Print losses every 10% of max_iterations
        #if iteration % (max_iterations // 20) == 0:
        if (iteration) % 50 == 0:
            #print('masks ',mask_one.sum(1),mask_zero.sum(1))
            print(output)
            #print('active_mask ', active_mask.sum())
            print(f"Iteration {iteration}, Total Loss: {total_loss.item()}")

    # Create masks for successful and unsuccessful attacks
    success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
    failure_mask = ~success_mask

    # Update upper bounds and constants where attacks are successful
    upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

    # Update lower bounds and constants where attacks are unsuccessful
    lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

    # Handle cases where upper bound is still too high
    upper_limit_mask = upper_bound < 1e9
    print(upper_limit_mask)
    CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
    CONST[failure_mask & ~upper_limit_mask] *= 10

    #print(f"Binary search step {outer_step}, CONST: {CONST}")
    print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")
    print(f"Binary search step {outer_step}, output: {model(o_bestattack)}")

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.05, binary_search_steps=5, confidence=0.1, initial_const=1, device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    targeted = True
    batch_size = imgs.shape[0]

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')

    for outer_step in range(binary_search_steps):

        if outer_step != (binary_search_steps - 1):
            lr = 0.01
        else :
            lr = learning_rate
            max_iterations = 6000
            print(f"Binary search step {outer_step}, max_iterations: {max_iterations}")

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=lr)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            #loss1 = criterion(output, active_labs_tensor)
            loss2 = l2dist
            loss = (active_CONST * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            #with torch.no_grad():
             #   l2norm = torch.linalg.norm(modifier.grad.data[active_mask], dim=-1, keepdim=True)
              #  modifier.grad.data[active_mask] = (modifier.grad.data[active_mask] / (l2norm + 1e-20)).float()

            optimizer.step()

            # Step the learning rate scheduler
            #scheduler.step()

            # Check for improvement in loss for each sample
            improved_mask = loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Update the best result found so far
            update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
            bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

            update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
            o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
            o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

            # Print losses every 10% of max_iterations
            #if iteration % (max_iterations // 20) == 0:
            #if (iteration) % 50 == 0:
                #print('masks ',mask_one.sum(1),mask_zero.sum(1))
                #print(output)
                #print('active_mask ', active_mask.sum())
                #print(f"Iteration {iteration}, Total Loss: {total_loss.item()}")

        # Create masks for successful and unsuccessful attacks
        success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
        failure_mask = ~success_mask

        # Update upper bounds and constants where attacks are successful
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

        # Update lower bounds and constants where attacks are unsuccessful
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        # Handle cases where upper bound is still too high
        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= 10

        # Set the precision for printing
        torch.set_printoptions(precision=2)
        #print(f"binary_search_steps {outer_step+1}, CONST: {CONST}")
        #print(model(o_bestattack))

    return o_bestattack

In [ ]:
def final_solution(oimgs, last_solution, model):
    perturbed = last_solution.clone().to(device)
    updated = oimgs.clone().to(device)
    totalchange2 = (last_solution - oimgs).abs()

    # Vectorized update of final tensor based on perturbation direction
    changes = torch.where((last_solution - oimgs) <= 0, torch.tensor(0.0, device=device), torch.tensor(1.0, device=device))

    for b in range(totalchange2.shape[0]):
        # Get sorted indices by descending change
        sorted_indices = torch.argsort(totalchange2[b], descending=True)

        for e in sorted_indices:
            if totalchange2[b, e] < 0.1:
                break

            # Update final[b, e] based on the direction of the perturbation
            updated[b, e] = changes[b, e]

            # Stop if the model's prediction is 0
            if model(updated[b].unsqueeze(0)).argmax() == 0:
                perturbed[b] = updated[b]
                break

    return perturbed

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print(len(mal_y_batch))
            print((y_pred == 1).sum().item())
            print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))
            print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
#CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.05, binary_search_steps=5, confidence=0.1, initial_const=1, device=device)
attack_params =  {'device':device, 'confidence':0.01, 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 0.05, 'initial_const': 1., 'binary_search_steps' : 5}
adv_predict(test_loader, combined_model, CW, **attack_params)

In [ ]:
#CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.05, binary_search_steps=5, confidence=0.1, initial_const=1, device=device)
attack_params =  {'device':device, 'confidence':0.01, 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 0.1, 'initial_const': 10., 'binary_search_steps' : 8}
adv_predict(test_loader, model_AT_rFGSM, CW, **attack_params)

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        loss = torch.clamp((real - other + 0.1), min=0.0)
        #loss = (real - other)

        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def pgd_min22(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + 0.1), min=0.0)
        loss = (real - other)

        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def pgd_min1(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        #batch_size = x_var.shape[0]
        #real = y_model[torch.arange(batch_size), y.view(-1).long()]
        #other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = (real - other)

        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print(len(mal_y_batch))
            print((y_pred == 1).sum().item())
            print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum(1))
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))
            print('***************************')



    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
#rounded
#loss = (real - other)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 5000, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd_min22, device, **attack_params)

In [ ]:
#rounded
#loss = torch.clamp((real - other + 0.1), min=0.0)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 5000, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min2, device, **attack_params)

In [ ]:
# loss = -criterion(y_model, torch.zeros_like(y.view(-1).long())) + 1
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 6000, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd, device, **attack_params)

In [ ]:
# loss = (real - other)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 1000, 'step_length': 0.5, 'norm': 'free', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min1, device, **attack_params)

In [ ]:
# loss = (real - other)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 1000, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min2, device, **attack_params)

In [ ]:
def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',rounding=False, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4insertion = (gradients >= 0) *(x_var < 1.)*  gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    #if random:
       #round_threshold = torch.rand(x_next.size())
    if rounding:
        x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next


def get_loss(x,y,model):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x)
    loss = criterion(outputs, y.view(-1).long())
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    return loss, done

def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

def PGD_Max2(x,y, model, insertion_array, removal_array, attack_list = ['linf', 'l2', 'l1'],k=50,steps_max=5, rounding=False, is_sample = False, varepsilon = 1e-20, is_report_loss_diff = False):
    """
    PGD_Max adversarial attack.

    Args:
        x: Input data tensor (shape: [samples, features])
        y: Ground truth labels tensor (shape: [samples])
        model: Neural network model
        attack_list: List of norms for attacks (default: ['linf', 'l2', 'l1'])
        steps_max: Maximum number of steps (default: 5)
        is_sample: Flag to sample randomly from the feasible area (default: False)
        vaρεpsilon: Tolerance for stopping condition (default: 1e-20)

    Returns:
        Adversarial version of input data (tensor)
    """
    batch_size = x.shape[0]
    norm_params = {
        'l1': {'k': k, 'step_length': 1.0,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff},
        'l2': {'k': k, 'step_length': 0.5,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff},
        'linf': {'k': k, 'step_length': 0.02,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff}
    }

    model.eval()
    with torch.no_grad():
        loss, done = get_loss(x,y,model) #shape:[samples],[samples]

    pre_loss = loss
    n = x.shape[0]
    adv_x = x.detach().clone().to(x.device)
    stop_flag = torch.zeros(n, dtype=torch.bool).to(x.device) #[samples]

    for t in range(steps_max):
      num_remaining  = (~stop_flag).sum().item()
      #print('number of remaining samples : ',num_remaining )
      if num_remaining  <= 0:
          break

      remaining_label = y[~stop_flag]
      pertbx = []

      for norm in attack_list:
          if norm in norm_params:
              params = norm_params[norm]
              perturbation = pgd_min2(adv_x[~stop_flag], remaining_label, model, insertion_array, removal_array, norm=norm, is_sample=is_sample, **params)
              #print("the number of added features : ", (perturbation.sum() - adv_x[~stop_flag].sum())/len(adv_x[~stop_flag]))
              pertbx.append(perturbation)
          else:
              raise ValueError("Expected 'l1' or 'l2' or 'linf' norm.")


      # here pertbx.shape = a list of (number of attacks  ,(num_remaining ,features))
      pertbx = torch.vstack(pertbx)
      # here pertbx.shape = a tensor (num_remaining *number of attacks samples, features)

      with torch.no_grad():
        remaining_label_ext = torch.cat([remaining_label] * len(attack_list)) #(labels*number of attacks )
        loss, done = get_loss(pertbx, remaining_label_ext,model) #(labels*number of attacks )

        # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
        # while if no attacks evade the victim successful, all perturbed examples are reminded for selection
        max_v = loss.amax()
        loss[done] += max_v

        loss = loss.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)
        done = done.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)

        success_flag = torch.any(done, dim=-1) #(num_remaining )

        pertbx = pertbx.reshape(len(attack_list), num_remaining , x.shape[1]).permute([1, 0, 2])#(num_remaining ,attacks,features)
        _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
        adv_x[~stop_flag] = pertbx[torch.arange(num_remaining ), indices]
        a_loss = loss[torch.arange(num_remaining ), indices]
        pre_stop_flag = stop_flag.clone()
        stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
        pre_loss[~pre_stop_flag] = a_loss

    return adv_x

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':1000, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}
ben_x_batch,mal_x_batch,pertb_mal_x = adv_predict(test_loader, combined_model2, PGD_Max2, device, **attack_params)

In [ ]:
import torch
import torch.optim as optim
import numpy as np
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
  for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(),dtype=torch.int64)


def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target


# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-4):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

model = model_AT_rFGSM
imgs = mal_x_batch
labs = target_label
targeted=True
abort_early=True
confidence=0.01
initial_const=1.
learning_rate=0.01
binary_search_steps=1
max_iterations=10000
removal_array = removal_array


batch_size = imgs.shape[0]

# Ensure fixed_features_mask is a tensor
if removal_array is not None:
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

# Calculate indices where mask is 0 or 1
mask_0 = non_fixed_features_mask == 0
mask_1 = non_fixed_features_mask == 1

# Create empty tensor for the result
imgs_tanh = torch.empty_like(imgs)

# Apply the transformations based on the mask
imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

# Use clone().detach() to copy the tensor correctly
imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Set the lower and upper bounds for binary search
lower_bound = torch.zeros(batch_size, device=device)
CONST = torch.full((batch_size,), initial_const, device=device)
upper_bound = torch.full((batch_size,), 1e10, device=device)

# Variables to store the best results
o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()

# Set model to evaluation mode
model.eval()



for outer_step in range(binary_search_steps):
    # Initialize modifier with requires_grad=True
    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

    # Setup optimizer
    #optimizer = optim.Adam([modifier], lr=learning_rate, betas=(0.5, 0.999))
    optimizer = optim.Adam([modifier], lr=learning_rate)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)
    prev_loss = torch.tensor(float('inf'), device=device)

    for iteration in range(max_iterations):

        optimizer.zero_grad()

        # Compute new images in original space
        modified_input = imgs_tensor + modifier
        newimg = torch.tanh(modified_input) * 0.5 + 0.5

        # Get model predictions (pre-softmax)
        output = model(newimg)

        # Compute L2 distance to original images
        l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

        # Calculate the loss based on predictions
        real = output[torch.arange(batch_size), labs_tensor]
        other = output[torch.arange(batch_size), (labs_tensor + (1 if targeted else -1))]


        # Calculate adversarial loss
        loss1 = torch.sum(CONST * torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0))
        #loss2 = torch.sum(l2dist / CONST )
        total_loss = loss1

        # Backpropagate the loss
        total_loss.backward()
        optimizer.step()

        # Apply the mask to reset the fixed features in the modifier
        #if removal_array is not None:
            #modifier.data = modifier.data * non_fixed_features_mask

        # Print losses every 10% of max_iterations
        if iteration % (max_iterations // 20) == 0:
            #print('real ',real)
            print(output)
            #print(f"Iteration {iteration}, Total Loss: {total_loss.item()}, Loss1: {loss1.item()}, Loss2: {loss2.item()}")
            print(f"Iteration {iteration}, Total Loss: {total_loss.item()}")

        # Abort early if no improvement
        if abort_early and iteration % 20 == 0:
            if loss1.item() > prev_loss * 0.9999:
                print('Early stopping: no improvement in loss')
                break
            prev_loss = total_loss.item()

        # Update the best result found so far
        update_mask = (l2dist < bestl2) & compare(output, labs_tensor, confidence, targeted)
        bestl2[update_mask] = l2dist[update_mask]
        bestscore[update_mask] = torch.argmax(output[update_mask], dim=1)

        update_mask2 = (l2dist < o_bestl2) & compare(output, labs_tensor, confidence, targeted)
        o_bestl2[update_mask2] = l2dist[update_mask2]
        o_bestscore[update_mask2] = torch.argmax(output[update_mask2], dim=1)
        o_bestattack[update_mask2] = newimg[update_mask2]

    # Create masks for successful and unsuccessful attacks
    success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
    failure_mask = ~success_mask

    # Update upper bounds and constants where attacks are successful
    upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

    # Update lower bounds and constants where attacks are unsuccessful
    lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

    # Handle cases where upper bound is still too high
    upper_limit_mask = upper_bound < 1e9
    CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
    CONST[failure_mask & ~upper_limit_mask] *= 10

    print(f"Binary search step {outer_step}, CONST: {CONST}")
    print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")
    print(f"Binary search step {outer_step}, output: {output}")



# Example usage of fixed_features_mask
# Let's say we want to fix some features in the input image, here is how you define the mask
# fixed_features_mask = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]]) # Example for a 3x3 image

In [ ]:
import torch
import torch.optim as optim
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
  for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(),dtype=torch.int64)


def torch_arctanh(x, eps=1e-5):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

model = model_AT_rFGSM
imgs = mal_x_batch[9:10]
labs = target_label[9:10]
targeted=True
abort_early=False
confidence=0.01
initial_const=1.
learning_rate=0.1
binary_search_steps=1
max_iterations=10000
removal_array = removal_array



batch_size = imgs.shape[0]

# Convert images to tanh-space
imgs_tanh = torch_arctanh(imgs)

# Use clone().detach() to copy the tensor correctly
imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Set the lower and upper bounds for binary search
lower_bound = torch.zeros(batch_size, device=device)
CONST = torch.full((batch_size,), initial_const, device=device)
upper_bound = torch.full((batch_size,), 1e10, device=device)

# Variables to store the best results
o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()

# Set model to evaluation mode
model.eval()

# Ensure fixed_features_mask is a tensor
if removal_array is not None:
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))
    print('imgs ', imgs.sum(1))
    print('fixed_features_mask ',(1 - non_fixed_features_mask).sum(1))
    #alpha = torch.full((mal_x_batch.shape), 1e6, device=device) * (1 - non_fixed_features_mask) + torch.ones_like(mal_x_batch, device=device)
    #print(alpha)

for outer_step in range(binary_search_steps):
    # Initialize modifier with requires_grad=True
    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

    # Setup optimizer
    optimizer = optim.SGD([modifier], lr=learning_rate)

    # Define custom gradient hook
    def grad_hook(grad):
        if removal_array is not None:
            return grad * non_fixed_features_mask
        return grad

    # Register the hook
    #modifier.register_hook(grad_hook)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)
    prev_loss = torch.tensor(float('inf'), device=device)

    for iteration in range(max_iterations):
        optimizer.zero_grad()

        # Compute new images in original space
        modified_input = imgs_tensor + modifier
        newimg = torch.tanh(modified_input) * 0.5 + 0.5

        # Get model predictions (pre-softmax)
        output = model(newimg)

        # Compute L2 distance to original images
        #l2dist = torch.sum(((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) * alpha) ** 2, dim=1)
        l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

        # Calculate the loss based on predictions
        real = output[torch.arange(batch_size), labs_tensor]
        other = output[torch.arange(batch_size), (labs_tensor + (1 if targeted else -1))]


        # Calculate adversarial loss
        loss1 = torch.sum(torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0))
        loss2 = torch.sum(l2dist)
        total_loss = loss1

        # Backpropagate the loss
        total_loss.backward()

        # Apply constraints using masks
        with torch.no_grad():
            grad_vars = -(modifier.grad.data)

            grad4insertion = (grad_vars >= 0) * grad_vars
            grad4removal = (grad_vars < 0) * non_fixed_features_mask * grad_vars
            grad_vars = grad4removal + grad4insertion

            # Apply projected gradient update
            modifier.data = modifier.data + grad_vars * learning_rate

        # Apply the mask to reset the fixed features in the modifier
        #if removal_array is not None:
            #modifier.data = modifier.data * non_fixed_features_mask

        # Print losses every 10% of max_iterations
        if iteration % (max_iterations // 10) == 0:
            #print('real ',real)
            print('output',output)
            print(f"Iteration {iteration}, Total Loss: {total_loss.item()}, Loss1: {loss1.item()}, Loss2: {loss2.item()}")

        # Abort early if no improvement
        if abort_early and iteration % (max_iterations // 10) == 0:
            if total_loss.item() > prev_loss * 0.999999:
                print('Early stopping: no improvement in loss')
                break
            prev_loss = total_loss.item()

        # Update the best result found so far
        update_mask = (l2dist < bestl2) & compare(output, labs_tensor, confidence, targeted)
        bestl2[update_mask] = l2dist[update_mask]
        bestscore[update_mask] = torch.argmax(output[update_mask], dim=1)

        update_mask2 = (l2dist < o_bestl2) & compare(output, labs_tensor, confidence, targeted)
        o_bestl2[update_mask2] = l2dist[update_mask2]
        o_bestscore[update_mask2] = torch.argmax(output[update_mask2], dim=1)
        o_bestattack[update_mask2] = newimg[update_mask2]

    # Create masks for successful and unsuccessful attacks
    success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
    failure_mask = ~success_mask

    # Update upper bounds and constants where attacks are successful
    upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

    # Update lower bounds and constants where attacks are unsuccessful
    lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

    # Handle cases where upper bound is still too high
    upper_limit_mask = upper_bound < 1e9
    CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
    CONST[failure_mask & ~upper_limit_mask] *= 10

    print(f"Binary search step {outer_step}, CONST: {CONST}")
    print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")



# Example usage of fixed_features_mask
# Let's say we want to fix some features in the input image, here is how you define the mask
# fixed_features_mask = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]]) # Example for a 3x3 image

In [ ]:
(abs(o_bestattack - mal_x_batch)).sum()

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print('outputs',outputs)
            print((torch.abs(pertb_mal_x - mal_x_batch)).sum())
            print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))
            break

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
attack_params =  {'device':device,'targeted':True, 'abort_early':True, 'confidence':0.01, 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 1., 'initial_const': 1., 'binary_search_steps' : 5}
adv_predict(test_loader, model_AT_rFGSM, CW, **attack_params)

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        if (t+1)%50 == 0:
            print(other)
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10000, 'step_length': 0.01, 'norm': 'free', 'is_report_loss_diff' : False}
adv_predict(test_loader, model_AT_rFGSM, pgd_min2, device, **attack_params)

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        loss = (real - other)
        if (t+1)%50 == 0:
            print(other)

        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
import torch
import torch.nn as nn

def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)

    # Initialize the optimizer for x_next
    x_next.requires_grad = True
    optimizer = torch.optim.Adam([x_next], lr=step_length)

    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        # Forward pass
        optimizer.zero_grad()
        y_model = model(x_next)

        batch_size = x_next.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long() - 1)]
        loss = (other - real)
        if (t+1)%50 == 0:
            print(y_model)

        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient and optimize
        loss.mean().backward()

        # Apply constraints using masks
        with torch.no_grad():
            grad_vars = x_next.grad.data

            grad4insertion = (grad_vars >= 0) * (x_next < 1.) * grad_vars
            grad4removal = (grad_vars < 0) * (x_next > 0.) * removal_array_updated * grad_vars
            grad_vars = grad4removal + grad4insertion

            if norm == 'linf':
                grad_vars = step_length * torch.sign(grad_vars).float()

            elif norm == 'l2':
                l2norm = torch.linalg.norm(grad_vars, dim=-1, keepdim=True)
                grad_vars = step_length * (grad_vars / (l2norm + 1e-20)).float()

            elif norm == 'free':
                grad_vars = grad_vars

            elif norm == 'l1':
                un_mod = torch.abs(x - x_next) <= 1e-6
                grad_vars = grad_vars * un_mod
                max_grad = torch.topk(torch.abs(grad_vars).view(grad_vars.size(0), -1), 1, dim=-1)[0]
                grad_vars = (torch.abs(grad_vars) >= max_grad).float() * torch.sign(grad_vars).float()
                done = get_done(x_next, y, model)
                if torch.all(done):
                    break
                grad_vars[done] = 0.

            else:
                raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

            # Apply projected gradient update
            x_next.data = torch.clamp(x_next.data + grad_vars * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    # x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next.data[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next.data - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next.data, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres', torch.mean(diff))

    return x_next.data

# ROUNDED

In [ ]:
def compare2(active_oimgs, newimg, model):
    # Ensure all tensors are on the GPU
    active_oimgs = active_oimgs.to(device)
    perturbed = newimg.to(device)

    l2dist = torch.sum((newimg -  active_oimgs) ** 2, dim=1)

    # Calculate the absolute difference once and keep everything on the GPU
    totalchange2 = (perturbed - active_oimgs).abs()

    # Create a mask for elements greater than 0.001
    significant_mask = totalchange2 > 0.2

    # Initialize results tensor
    results = torch.zeros(active_oimgs.shape[0], dtype=torch.bool, device=device)

    # Prepare a tensor to apply changes in one go
    final = active_oimgs.clone()

    # Vectorized update of final tensor based on perturbation direction
    changes = torch.where((perturbed - active_oimgs) <= 0, torch.tensor(0.0, device=device), torch.tensor(1.0, device=device))

    for b in range(totalchange2.shape[0]):

        # Get sorted indices for the significant changes only
        sorted_idx = torch.argsort(totalchange2[b][significant_mask[b]], descending=True)

        # Retrieve the original indices that correspond to the sorted significant changes
        relevant_indices = torch.nonzero(significant_mask[b], as_tuple=True)[0][sorted_idx]

        # Update only the significant changes
        for e in relevant_indices:

            # Apply the change to the final tensor
            final[b, e] = changes[b, e]

            # Check the model's prediction
            if model(final[b].unsqueeze(0)).argmax() == 0:
                results[b] = True
                break

    return results, final, l2dist_updated

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.05, binary_search_steps=5, confidence=0.1, initial_const=1, device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    targeted = True
    batch_size = imgs.shape[0]

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction='none')

    for outer_step in range(binary_search_steps):
        print('outer_step ',outer_step)
        if outer_step != (binary_search_steps - 1):
            lr = 0.01
        else :
            lr = learning_rate
            max_iterations = 6000
            print(f"Binary search step {outer_step}, max_iterations: {max_iterations}")

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=lr)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)
        bestattack = imgs.clone().detach()

        for iteration in range(max_iterations):
            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]
            active_oimgs = imgs[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  active_oimgs) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            #loss1 = criterion(output, active_labs_tensor)
            loss2 = l2dist
            loss = (active_CONST * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            #with torch.no_grad():
             #   l2norm = torch.linalg.norm(modifier.grad.data[active_mask], dim=-1, keepdim=True)
              #  modifier.grad.data[active_mask] = (modifier.grad.data[active_mask] / (l2norm + 1e-20)).float()

            optimizer.step()

            # Step the learning rate scheduler
            #scheduler.step()

            if (iteration + 1) % 25 == 0:
                # Check for improvement in loss for each sample
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

                # Update the best result found so far
                #update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                comparing_result, final, l2dist_updated = compare2(active_oimgs, newimg, model)
                final_output = model(final)
                update_mask = (l2dist_updated < bestl2[active_mask]) & comparing_result
                bestl2[active_mask] = torch.where(update_mask, l2dist_updated, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(final_output, dim=1), bestscore[active_mask])
                bestattack[active_mask] = torch.where(update_mask.unsqueeze(-1), newimg, bestattack[active_mask])

                #update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                update_mask2 = (l2dist_updated < o_bestl2[active_mask]) & comparing_result
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist_updated, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(final_output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

            # Print losses every 10% of max_iterations
            #if iteration % (max_iterations // 20) == 0:
            #if (iteration) % 50 == 0:
                #print('masks ',mask_one.sum(1),mask_zero.sum(1))
                #print(output)
                #print('active_mask ', active_mask.sum())
                #print(f"Iteration {iteration}, Total Loss: {total_loss.item()}")

        # Create masks for successful and unsuccessful attacks
        comparing_result, final, l2dist_updated = compare2(imgs, bestattack, model)
        success_mask = (bestscore != -1) & comparing_result
        print('success_mask.sum() : ',success_mask.sum())
        failure_mask = ~success_mask

        # Update upper bounds and constants where attacks are successful
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

        # Update lower bounds and constants where attacks are unsuccessful
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        # Handle cases where upper bound is still too high
        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= 10

        # Set the precision for printing
        torch.set_printoptions(precision=2)
        #print(f"binary_search_steps {outer_step+1}, CONST: {CONST}")
        #print(model(o_bestattack))

    return o_bestattack

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            #pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
            pertb_mal_x = final_solution(mal_x_batch, pertb_mal_x, model)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print(len(mal_y_batch))
            print((y_pred == 1).sum().item())
            print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))
            print('***************************')

            break


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
#CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.05, binary_search_steps=5, confidence=0.1, initial_const=1, device=device)
attack_params =  {'device':device, 'confidence':100, 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 0.01, 'initial_const': 10., 'binary_search_steps' : 8}
adv_predict(test_loader, combined_model, CW, **attack_params)

In [ ]:
#CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.05, binary_search_steps=5, confidence=0.1, initial_const=1, device=device)
attack_params =  {'device':device, 'confidence':100., 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 0.01, 'initial_const': 0.01, 'binary_search_steps' : 8}
adv_predict(test_loader, combined_model, CW, **attack_params)

## Part 3 — The CW-L0 attack

### Step 5 — The packaged helpers (from the 222 notebook)

# Carlini and Wagner L0 attack

In [ ]:
def compare2_old(active_oimgs, newimg, model,active_valid):
    # Ensure all tensors are on the GPU
    active_oimgs = active_oimgs.to(device)
    perturbed = newimg.to(device)

    # Calculate the absolute difference once and keep everything on the GPU
    totalchange2 = (perturbed - active_oimgs).abs()

    # Sort indices along the last dimension and get the sorted tensor
    sorted_indices = torch.argsort(totalchange2, dim=1, descending=True)

    # Initialize results tensor
    results = torch.zeros(active_oimgs.shape[0], dtype=torch.bool, device=device)

    # Prepare a tensor to apply changes in one go
    final = active_oimgs.clone()

    # Vectorized update of final tensor based on perturbation direction
    changes = torch.where(perturbed - active_oimgs < 0, torch.tensor(0.0, device=device), torch.tensor(1.0, device=device))

    for b in range(totalchange2.shape[0]):
        # Select the sorted indices for the current sample
        sorted_idx = sorted_indices[b]

        # Update only the significant changes
        for e in sorted_idx:
            if totalchange2[b, e] < 0.2:
                break

            # Apply the change to the final tensor
            final[b, e] = changes[b, e]

            # Check the model's prediction in a vectorized way
            if model(final[b].unsqueeze(0)).argmax() == 0:
                results[b] = True
                break

    return results

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

In [ ]:
def compare2(active_oimgs, newimg, model,active_valid):
    # Ensure all tensors are on the GPU
    active_oimgs = active_oimgs.to(device)
    perturbed = newimg.to(device)

    # Calculate the absolute difference once and keep everything on the GPU
    totalchange2 = (perturbed - active_oimgs).abs()

    # Create a mask for elements greater than 0.001
    significant_mask = totalchange2 > 0.2

    # Initialize results tensor
    results = torch.zeros(active_oimgs.shape[0], dtype=torch.bool, device=device)

    # Prepare a tensor to apply changes in one go
    final = active_oimgs.clone()

    # Vectorized update of final tensor based on perturbation direction
    changes = torch.where((perturbed - active_oimgs) <= 0, torch.tensor(0.0, device=device), torch.tensor(1.0, device=device))

    for b in range(totalchange2.shape[0]):

        # Get sorted indices for the significant changes only
        sorted_idx = torch.argsort(totalchange2[b][significant_mask[b]], descending=True)

        # Retrieve the original indices that correspond to the sorted significant changes
        relevant_indices = torch.nonzero(significant_mask[b], as_tuple=True)[0][sorted_idx]

        # Update only the significant changes
        for e in relevant_indices:

            # Apply the change to the final tensor
            final[b, e] = changes[b, e]

            # Check the model's prediction
            if model(final[b].unsqueeze(0)).argmax() == 0:
                results[b] = True
                break

    return results

In [ ]:
def evaluation(oimgs,last_solution,model):
    pertb_mal_x = final_solution(oimgs,last_solution,model)

    n_samples = len(oimgs)

    outputs = model(oimgs)
    predicted = outputs.argmax(1)
    cor_test = (predicted == 1).sum().item()

    outputs = model(pertb_mal_x)
    y_pred = outputs.argmax(1)
    cor_ad_test = (y_pred == 1).sum().item()

    print('-----------------------------------------------------------------------')
    print('number of samples : ',n_samples)
    print('number of successful attack : ',(y_pred == 0).sum().item())
    print('Number of updated features in each sample : ',(torch.abs(pertb_mal_x - oimgs)).sum(1))
    print('Number of updated features  : ',(torch.abs(pertb_mal_x - oimgs)).sum())
    print('Mean number of updated features : ',(torch.abs(pertb_mal_x - oimgs)).sum()/(y_pred == 0).sum().item())
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")
    print('-----------------------------------------------------------------------')

# Combine model

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.6)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

    def forward_from_z(self, z):
        output = self.fc3(z)
        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

In [ ]:
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

combined_model = Wrapper(core_model, proj_head).to(device)
combined_model.load_state_dict(torch.load('model_AT_max%20(4).pth', map_location=torch.device(device)))

In [ ]:
with torch.no_grad():
    i = 0
    for x_test, y_test in test_loader:
        i += 1
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        if i == 1:
            break
print('mal_x_batch ',mal_x_batch.shape)

model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.01
initial_const = 1e-3
largest_const = 2e9
targeted = True
confidence=0.01

oimgs = mal_x_batch
lab = mal_y_batch
batch_size = oimgs.shape[0]

labs = torch.zeros_like(lab.squeeze(), dtype=torch.int64)

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

#prev = oimgs.clone()
#last_solution = None
last_solution = oimgs.clone().detach().to(dtype=torch.float32, device=device)
const = initial_const

oimgs = oimgs.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Track active samples, initial active mask
active_mask_i = torch.ones(batch_size, dtype=torch.bool, device=device)
# if model predict wrongly, no need to perturb
active_mask_i[compare(model(oimgs), labs_tensor, confidence, targeted)] = False



CONST = [0.001,0.005,0.01,0.05,0.1,0.5,1.,5.,50.,1e4,1e9,largest_const]
i = CONST.index(const)

while const<largest_const:
    model.eval()
    #initializing active mask for a const
    active_mask = active_mask_i.clone().detach().to(dtype=torch.bool, device=device)

    # number of updates in the last iteration
    j = 0

    while True:
        print('CONST ',const)

        pre_active_mask = active_mask.clone().detach().to(dtype=torch.bool, device=device)

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)


        # Calculate indices where mask is 0 or 1
        mask_0 = valid == 0
        mask_1 = valid == 1

        # Create empty tensor for the result
        imgs_tanh = torch.empty_like(oimgs)

        # Apply the transformations based on the mask
        imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
        imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

        # Use clone().detach() to copy the tensor correctly
        imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            j += 1
            print(j,'th update')
            learning_rate = 0.01
            max_iterations = 6000
            # IMPORTANT: final round we increase confidence to go further , before we observed
            # pgdmin1(without constraint with confidence) got way result than pgdmin2 which has confidence
            # it affects just samples without change untile last round



            #print('learning_rate ',learning_rate)
            #print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)

        for step in range(max_iterations):

            if active_mask.sum() == 0:
              break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_oimgs = oimgs[active_mask]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_valid = valid[active_mask]
            #active_CONST = CONST[active_mask]
            active_modifier.retain_grad()

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            if const != 1e9:
                loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
                loss2 = l2dist
                loss = (const * loss1) + loss2
                total_loss = torch.sum(loss)
            else :
                loss1 = (other - real)
                total_loss = torch.sum(loss1)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            optimizer.zero_grad()


            # Check for improvement in loss for each sample
            improved_mask = total_loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, total_loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            if (no_improvement_count[active_mask.clone()] > 100).sum() == (active_mask.sum()):
                print('BREAK')
                break
            #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

            #with torch.no_grad():
            if (step + 1) % 40 == 0:
                active_mask2 = compare2(active_oimgs,newimg,model,active_valid)
                if active_mask2.sum():
                    print(active_mask2.sum())
                    #print('output2 ',output2[active_mask2])
                    #print(modifier.grad.detach().sum())
                    gradientnorm, scores, nimg = active_modifier.grad[active_mask2].detach(), output[active_mask2].detach(), newimg[active_mask2].detach()

                    equal_count = active_oimgs[active_mask2].shape[-1] - torch.sum(torch.abs(active_oimgs[active_mask2] - nimg) < 1e-4,1)
                    #print("Fixed features:", torch.sum(1 - active_valid[active_mask2],1), "Equal count:", equal_count)

                    #if torch.sum(valid) == 0:
                        #last_solution = oimgs
                        #break

                    valid0 = active_valid[active_mask2]
                    #print('valid0',valid0.sum())
                    totalchange = (nimg - active_oimgs[active_mask2]).abs() * gradientnorm.abs()

                    totalchange = totalchange.flatten(1)

                    for b in range(totalchange.shape[0]):
                        sorted_idx = torch.argsort(totalchange[b])
                        for e in sorted_idx:
                            if valid0[b, e]:
                                valid0[b, e] = 0
                                if totalchange[b, e] > 0.01:
                                    break


                    #print('valid0',valid0.sum())

                    '''for e in torch.argsort(totalchange):
                        if valid0[e]:
                            valid0[e] = 0
                            if totalchange[e] > 0.01 :
                                break'''

                    active_valid[active_mask2] = valid0.view((active_oimgs[active_mask2].shape))
                    valid[active_mask] = active_valid
                    #print("Now Fixed features:", torch.sum((1 - valid),1))

                    #update active mask
                    #active_mask[active_mask][active_mask2] = False
                    # Step 1: Identify the indices where active_mask is True
                    true_indices = torch.nonzero(active_mask, as_tuple=True)[0]  # This gives the indices [0, 1, 2]

                    # Step 2: Apply active_mask2 to modify these specific indices
                    active_mask[true_indices[active_mask2]] = False
                    #print('last avtive mask ',active_mask)

                    #last_solution = prev = nimg
                    last_solution[true_indices[active_mask2]] = nimg

        if const == 1e9 and j > 10:
            const = CONST[-1]
            print('BREAK, due to to too much updates')
            break

        # Perform the XOR operation(which samples update)
        result = torch.logical_xor(active_mask,pre_active_mask)
        #print(result.sum())
        if result.sum() != 0 :
            active_mask = result.clone().detach().to(dtype=torch.bool, device=device)

        else :
            i += 1
            const = CONST[i]
            if const == 1e9:
                evaluation(oimgs,last_solution,model)
            break


evaluation(oimgs,last_solution,model)

In [ ]:
with torch.no_grad():
    i = 0
    for x_test, y_test in test_loader:
        i += 1
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        if i == 1:
            break
print('mal_x_batch ',mal_x_batch.shape)

model = combined_model.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.01
initial_const = 1e-3
largest_const = 2e9
targeted = True
confidence=0.01

oimgs = mal_x_batch
lab = mal_y_batch
batch_size = oimgs.shape[0]

labs = torch.zeros_like(lab.squeeze(), dtype=torch.int64)

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

#prev = oimgs.clone()
#last_solution = None
last_solution = oimgs.clone().detach().to(dtype=torch.float32, device=device)
const = initial_const

oimgs = oimgs.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Track active samples, initial active mask
active_mask_i = torch.ones(batch_size, dtype=torch.bool, device=device)
# if model predict wrongly, no need to perturb
active_mask_i[compare(model(oimgs), labs_tensor, confidence, targeted)] = False



CONST = [0.001,0.005,0.01,0.05,0.1,0.5,1.,5.,50.,1e4,1e9,largest_const]
i = CONST.index(const)

while const<largest_const:
    model.eval()
    #initializing active mask for a const
    active_mask = active_mask_i.clone().detach().to(dtype=torch.bool, device=device)

    # number of updates in the last iteration
    j = 0

    while True:
        print('CONST ',const)

        pre_active_mask = active_mask.clone().detach().to(dtype=torch.bool, device=device)

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)


        # Calculate indices where mask is 0 or 1
        mask_0 = valid == 0
        mask_1 = valid == 1

        # Create empty tensor for the result
        imgs_tanh = torch.empty_like(oimgs)

        # Apply the transformations based on the mask
        imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
        imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

        # Use clone().detach() to copy the tensor correctly
        imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            j += 1
            print(j,'th update')
            learning_rate = 0.01
            max_iterations = 6000
            # IMPORTANT: final round we increase confidence to go further , before we observed
            # pgdmin1(without constraint with confidence) got way result than pgdmin2 which has confidence
            # it affects just samples without change untile last round



            #print('learning_rate ',learning_rate)
            #print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)

        for step in range(max_iterations):

            if active_mask.sum() == 0:
              break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_oimgs = oimgs[active_mask]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_valid = valid[active_mask]
            #active_CONST = CONST[active_mask]
            active_modifier.retain_grad()

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            if const != 1e9:
                loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
                loss2 = l2dist
                loss = (const * loss1) + loss2
                total_loss = torch.sum(loss)
            else :
                loss1 = (other - real)
                total_loss = torch.sum(loss1)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            optimizer.zero_grad()


            # Check for improvement in loss for each sample
            improved_mask = total_loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, total_loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            if (no_improvement_count[active_mask.clone()] > 100).sum() == (active_mask.sum()):
                print('BREAK')
                break
            #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

            #with torch.no_grad():
            if (step + 1) % 40 == 0:
                active_mask2 = compare2(active_oimgs,newimg,model,active_valid)
                if active_mask2.sum():
                    print(active_mask2.sum())
                    #print('output2 ',output2[active_mask2])
                    #print(modifier.grad.detach().sum())
                    gradientnorm, scores, nimg = active_modifier.grad[active_mask2].detach(), output[active_mask2].detach(), newimg[active_mask2].detach()

                    equal_count = active_oimgs[active_mask2].shape[-1] - torch.sum(torch.abs(active_oimgs[active_mask2] - nimg) < 1e-4,1)
                    #print("Fixed features:", torch.sum(1 - active_valid[active_mask2],1), "Equal count:", equal_count)

                    #if torch.sum(valid) == 0:
                        #last_solution = oimgs
                        #break

                    valid0 = active_valid[active_mask2]
                    #print('valid0',valid0.sum())
                    totalchange = (nimg - active_oimgs[active_mask2]).abs() * gradientnorm.abs()

                    totalchange = totalchange.flatten(1)

                    for b in range(totalchange.shape[0]):
                        sorted_idx = torch.argsort(totalchange[b])
                        for e in sorted_idx:
                            if valid0[b, e]:
                                valid0[b, e] = 0
                                if totalchange[b, e] > 0.01:
                                    break


                    #print('valid0',valid0.sum())

                    '''for e in torch.argsort(totalchange):
                        if valid0[e]:
                            valid0[e] = 0
                            if totalchange[e] > 0.01 :
                                break'''

                    active_valid[active_mask2] = valid0.view((active_oimgs[active_mask2].shape))
                    valid[active_mask] = active_valid
                    #print("Now Fixed features:", torch.sum((1 - valid),1))

                    #update active mask
                    #active_mask[active_mask][active_mask2] = False
                    # Step 1: Identify the indices where active_mask is True
                    true_indices = torch.nonzero(active_mask, as_tuple=True)[0]  # This gives the indices [0, 1, 2]

                    # Step 2: Apply active_mask2 to modify these specific indices
                    active_mask[true_indices[active_mask2]] = False
                    #print('last avtive mask ',active_mask)

                    #last_solution = prev = nimg
                    last_solution[true_indices[active_mask2]] = nimg

        if const == 1e9 and j > 10:
            const = CONST[-1]
            print('BREAK, due to to too much updates')
            break

        # Perform the XOR operation(which samples update)
        result = torch.logical_xor(active_mask,pre_active_mask)
        #print(result.sum())
        if result.sum() != 0 :
            active_mask = result.clone().detach().to(dtype=torch.bool, device=device)

        else :
            i += 1
            const = CONST[i]
            if const == 1e9:
                evaluation(oimgs,last_solution,model)
            break


evaluation(oimgs,last_solution,model)

### Step 6 — The L0 runs: iteratively fixing the most important features

# Carlini and Wagner L0 attack

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

In [ ]:
import torch.optim as optim
import numpy as np

# Constants
max_iterations = 1000
abort_early = False
learning_rate = 0.1
initial_const = 1e-4
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01


def gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, const, largest_const,
                     const_factor, oimgs, labs, valid, device):

    # Convert input arrays to tensors
    oimgs = torch.tensor(oimgs, dtype=torch.float32, device=device)
    labs_tensor = torch.tensor(labs, dtype=torch.int64, device=device)
    #starts = torch.tensor(starts, dtype=torch.float32, device=device)

    # Calculate indices where mask is 0 or 1
    mask_0 = valid == 0
    mask_1 = valid == 1

    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(oimgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

    #valid = torch.tensor(non_fixed_features_mask, dtype=torch.float32, device=device)


    model.eval()

    CONST = [initial_const, 0.01,0.05,0.1,0.5,1.,10.,1000.,1e9,largest_const]
    i = CONST.index(const)

    #for const in CONST:
    while const < largest_const:
        print('CONST ',const)

        best_loss = torch.tensor(float('inf'), device=device)
        no_improvement_counter = 0  # Counter for no improvement

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            learning_rate=0.001
            max_iterations=5000
            print('learning_rate ',learning_rate)
            print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)

        for step in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5
            output = model(newimg)

            #l2dist = torch.sum((newimg - oimgs) ** 2, dim=1)
            l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

            # Calculate the loss based on predictions
            real = output[0, labs_tensor]
            other = output[0, (labs_tensor + (1 if targeted else -1))]


            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            loss2 = l2dist
            loss = (const * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()
            optimizer.step()

            #if step % ((max_iterations+1) // 50) == 0 :
                #print('output ',output)
                #print(modifier.grad.detach().sum())
                #print(compare(output, labs_tensor, confidence, targeted))

            # Check if the loss has improved
            if total_loss.item() < best_loss:
                best_loss = total_loss.item()
                no_improvement_counter = 0  # Reset the counter if improvement is found
            else:
                no_improvement_counter += 1  # Increment the counter if no improvement

            if abort_early and (no_improvement_counter >= 100):
                print(f"No improvement for 100 consecutive iterations at step {step}. Breaking out of loop.")
                break

            #if abort_early and step % (max_iterations // 10) == 0 and total_loss.item() > prev_loss * .999999:
                #break

            prev_loss = total_loss.item()

            with torch.no_grad():
                result = newimg.clone()
                result[result<0.5] = 0.
                result[result>0.5] = 1.
                output2 = model(result)

            if compare(output, labs_tensor, confidence, targeted):
                print('output ',output)
                print('OK')

            if compare(output2, labs_tensor, confidence, targeted):
            #if compare(output, labs_tensor, confidence, targeted):
                print('output ',output)
                print('output2 ',output2)
                print(modifier.grad.detach().sum())
                return modifier.grad.detach().cpu(), output.detach().cpu(), newimg.detach().cpu(), const

        i += 1
        const = CONST[i]

    return None

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

oimgs = mal_x_batch[:1]
labs = target_label[:1]
model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.05
initial_const = 1e-3
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

prev = oimgs.clone()
last_solution = None
const = initial_const

while True:
    res = gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, const, largest_const,
                          const_factor, oimgs, labs, valid, device)
    if res is None:
        print("Final answer")
        break

    gradientnorm, scores, nimg, const = res
    if REDUCE_CONST:
        const /= 2.

    print(torch.sum(torch.abs(oimgs - nimg) < 1e-4).item())
    equal_count = oimgs.shape[-1] - torch.sum(torch.abs(oimgs - nimg) < 1e-4).item()
    print("Fixed features:", torch.sum(1 - valid).item(), "Equal count:", equal_count)

    if torch.sum(valid) == 0:
        last_solution = [oimgs]
        break


    valid = valid.flatten()
    totalchange = (nimg - oimgs).abs() * gradientnorm[0].abs()

    totalchange = totalchange.flatten()

    did = 0
    for e in torch.argsort(totalchange):
        if valid[e]:
            did += 1
            valid[e] = 0

            if totalchange[e] > 0.01 :
                break

    valid = valid.view((oimgs.shape))
    print("Now Fixed features:", torch.sum(1 - valid).item())

    last_solution = prev = nimg
    break

In [ ]:
torch.sum(torch.abs(last_solution - oimgs))

In [ ]:
result = nimg.clone()
result[result<0.5] = 0.
result[result>0.5] = 1.
compare(result, labs, confidence, targeted)

In [ ]:
model_AT_rFGSM(result)

In [ ]:
torch.sum(torch.abs(result - oimgs))

In [ ]:
import torch.optim as optim
import numpy as np

# Constants
max_iterations = 1000
abort_early = False
learning_rate = 0.1
initial_const = 1e-4
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01


def gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, const, largest_const,
                     const_factor, oimg, labs, valid, device):

    # Convert input arrays to tensors
    oimgs = oimg.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)
    #oimgs = torch.tensor(oimgs, dtype=torch.float32, device=device)
    #labs_tensor = torch.tensor(labs, dtype=torch.int64, device=device)

    #starts = torch.tensor(starts, dtype=torch.float32, device=device)

    # Calculate indices where mask is 0 or 1
    mask_0 = valid == 0
    mask_1 = valid == 1

    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(oimgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

    #valid = torch.tensor(non_fixed_features_mask, dtype=torch.float32, device=device)


    model.eval()

    CONST = [initial_const, 0.01,0.05,0.1,0.5,1.,10.,1000.,1e9,largest_const]
    i = CONST.index(const)

    #for const in CONST:
    while const < largest_const:
        #print('CONST ',const)

        best_loss = torch.tensor(float('inf'), device=device)
        no_improvement_counter = 0  # Counter for no improvement

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            learning_rate=0.01
            max_iterations=5000
            #print('learning_rate ',learning_rate)
            #print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)

        for step in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5
            output = model(newimg)

            #l2dist = torch.sum((newimg - oimgs) ** 2, dim=1)
            l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

            # Calculate the loss based on predictions
            real = output[0, labs_tensor]
            other = output[0, (labs_tensor + (1 if targeted else -1))]


            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            loss2 = l2dist
            loss = (const * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()
            optimizer.step()

            #if step % ((max_iterations+1) // 50) == 0 :
                #print('output ',output)
                #print(modifier.grad.detach().sum())
                #print(compare(output, labs_tensor, confidence, targeted))

            # Check if the loss has improved
            if total_loss.item() < best_loss:
                best_loss = total_loss.item()
                no_improvement_counter = 0  # Reset the counter if improvement is found
            else:
                no_improvement_counter += 1  # Increment the counter if no improvement

            if abort_early and (no_improvement_counter >= 100):
                #print(f"No improvement for 100 consecutive iterations at step {step}. Breaking out of loop.")
                break

            #if abort_early and step % (max_iterations // 10) == 0 and total_loss.item() > prev_loss * .999999:
                #break

            prev_loss = total_loss.item()

            with torch.no_grad():
                result = newimg.clone()
                result[result<0.5] = 0.
                result[result>0.5] = 1.
                output2 = model(result)

            '''if compare(output, labs_tensor, confidence, targeted):
                print('output ',output)
                print('OK')'''


            if compare(output2, labs_tensor, confidence, targeted):
            #if compare(output, labs_tensor, confidence, targeted):
                #print('output ',output)
                #print('output2 ',output2)
                #print(modifier.grad.detach().sum())
                return modifier.grad.detach().cpu(), output.detach().cpu(), newimg.detach().cpu(), const
        i += 1
        const = CONST[i]

    return None

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break


model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.05
initial_const = 1e-3
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01

for i,(oimgs,lab) in enumerate(zip(mal_x_batch[11:], mal_y_batch[11:])):
    print("Attack sample ",i)
    oimgs.unsqueeze_(0)
    labs = torch.zeros_like(lab.squeeze(), dtype=torch.int64)

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
    valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
    print('oimgs ', oimgs.sum(1))
    print('fixed_features_mask ',(1 - valid).sum(1))

    prev = oimgs.clone()
    last_solution = None
    const = initial_const

    # if model predict wrongly, no need to perturb
    if compare(model(oimgs), labs, confidence, targeted):
        print('output = ',model(oimgs))
        print('L0 = ',torch.sum(torch.abs(oimgs - oimgs)))
        print('------------------------------------')
        continue

    while True:
        res = gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, const, largest_const,
                              const_factor, oimgs, labs, valid, device)
        if res is None:
            print("Final answer : ")
            break

        gradientnorm, scores, nimg, const = res
        if REDUCE_CONST:
            const /= 2.

        equal_count = oimgs.shape[-1] - torch.sum(torch.abs(oimgs - nimg) < 1e-4).item()
        #print("Fixed features:", torch.sum(1 - valid).item(), "Equal count:", equal_count)

        if torch.sum(valid) == 0:
            last_solution = oimgs
            break


        valid = valid.flatten()
        totalchange = (nimg - oimgs).abs() * gradientnorm[0].abs()

        totalchange = totalchange.flatten()

        for e in torch.argsort(totalchange):
            if valid[e]:
                valid[e] = 0
                if totalchange[e] > 0.01 :
                    break

        valid = valid.view((oimgs.shape))
        #print("Now Fixed features:", torch.sum(1 - valid).item())

        last_solution = prev = nimg

    if last_solution!=None:
        result = last_solution.clone()
        result[result<0.5] = 0.
        result[result>0.5] = 1.
        compare(result, labs, confidence, targeted)
        print('output = ',model(result))
        print('L0 = ',torch.sum(torch.abs(result - oimgs)))
        print('------------------------------------')
    else :
        print('Attack failed')
        print('------------------------------------')

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break


model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.05
initial_const = 1e-3
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01

oimgs = mal_x_batch[:2]
lab = mal_y_batch[:2]
batch_size = oimgs.shape[0]

labs = torch.zeros_like(lab.squeeze(), dtype=torch.int64)

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

prev = oimgs.clone()
last_solution = None
const = initial_const

oimgs = oimgs.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Track active samples
active_mask0 = torch.ones(batch_size, dtype=torch.bool, device=device)
# if model predict wrongly, no need to perturb
active_mask0[compare(model(oimgs), labs_tensor, confidence, targeted)] = False

while const<largest_const:
    model.eval()

    CONST = [initial_const, 0.01,0.05,0.1,0.5,1.,10.,1000.,1e9,largest_const]
    i = CONST.index(const)

    #for const in CONST:
    while const < largest_const:
        print('CONST ',const)

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)
        active_mask = active_mask0.clone().detach().to(dtype=torch.bool, device=device)

        # Calculate indices where mask is 0 or 1
        mask_0 = valid == 0
        mask_1 = valid == 1

        # Create empty tensor for the result
        imgs_tanh = torch.empty_like(oimgs)

        # Apply the transformations based on the mask
        imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
        imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

        # Use clone().detach() to copy the tensor correctly
        imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            learning_rate=0.01
            max_iterations=5000
            #print('learning_rate ',learning_rate)
            #print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)
        for step in range(max_iterations):

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_oimgs = oimgs[active_mask]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_valid = valid[active_mask]
            #active_CONST = CONST[active_mask]
            active_modifier.retain_grad()

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            #loss1 = criterion(output, active_labs_tensor)
            loss2 = l2dist
            loss = (const * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            optimizer.zero_grad()


            # Check for improvement in loss for each sample
            improved_mask = loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100


            with torch.no_grad():
                result = newimg.clone()
                result[result<0.5] = 0.
                result[result>0.5] = 1.
                output2 = model(result)

            '''if compare(output, labs_tensor, confidence, targeted):
                print('output ',output)
                print('OK')'''

            if compare(output2, active_labs_tensor, confidence, targeted).sum():
                active_mask2 = compare(output2, active_labs_tensor, confidence, targeted)
                print(active_mask2)

                print('output2 ',output2[active_mask2])
                #print(modifier.grad.detach().sum())
                gradientnorm, scores, nimg = active_modifier.grad[active_mask2].detach(), output[active_mask2].detach(), newimg[active_mask2].detach()

                equal_count = active_oimgs[active_mask2].shape[-1] - torch.sum(torch.abs(active_oimgs[active_mask2] - nimg) < 1e-4,1)
                print("Fixed features:", torch.sum(1 - active_valid[active_mask2],1), "Equal count:", equal_count)

                #if torch.sum(valid) == 0:
                    #last_solution = oimgs
                    #break

                valid0 = active_valid[active_mask2]
                print('valid0',valid0.sum())
                totalchange = (nimg - active_oimgs[active_mask2]).abs() * gradientnorm.abs()

                totalchange = totalchange.flatten(1)
                print('totalchange', totalchange.shape)

                for b in range(totalchange.shape[0]):
                    for e in torch.argsort(totalchange[b]):
                        if valid0[b, e]:
                            valid0[b, e] = 0
                            if totalchange[b, e] > 0.01:
                                break

                print('valid0',valid0.sum())

                '''for e in torch.argsort(totalchange):
                    if valid0[e]:
                        valid0[e] = 0
                        if totalchange[e] > 0.01 :
                            break'''

                active_valid[active_mask2] = valid0.view((active_oimgs[active_mask2].shape))
                valid[active_mask] = active_valid
                print("Now Fixed features:", torch.sum((1 - valid),1))

                last_solution = prev = nimg

                i -= 1
                break

        i += 1
        const = CONST[i]



if last_solution!=None:
    result = last_solution.clone()
    result[result<0.5] = 0.
    result[result>0.5] = 1.
    compare(result, labs, confidence, targeted)
    print('output = ',model(result))
    print('L0 = ',torch.sum(torch.abs(result - oimgs),1))
    print('------------------------------------')
else :
    print('Attack failed')
    print('------------------------------------')

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break


model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.05
initial_const = 1e-3
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01

oimgs = mal_x_batch[6:8]
lab = mal_y_batch[6:8]
batch_size = oimgs.shape[0]

labs = torch.zeros_like(lab.squeeze(), dtype=torch.int64)

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

#prev = oimgs.clone()
#last_solution = None
last_solution = oimgs.clone().detach().to(dtype=torch.float32, device=device)
const = initial_const

oimgs = oimgs.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Track active samples, initial active mask
active_mask_i = torch.ones(batch_size, dtype=torch.bool, device=device)
# if model predict wrongly, no need to perturb
active_mask_i[compare(model(oimgs), labs_tensor, confidence, targeted)] = False

CONST = [initial_const,0.005,0.01,0.05,0.1,0.5,1.,10.,1000.,1e9,largest_const]
i = CONST.index(const)

while const<largest_const:
    model.eval()
    #initializing active mask for a const
    active_mask = active_mask_i.clone().detach().to(dtype=torch.bool, device=device)

    while True:
        print('CONST ',const)

        pre_active_mask = active_mask.clone().detach().to(dtype=torch.bool, device=device)

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)


        # Calculate indices where mask is 0 or 1
        mask_0 = valid == 0
        mask_1 = valid == 1

        # Create empty tensor for the result
        imgs_tanh = torch.empty_like(oimgs)

        # Apply the transformations based on the mask
        imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
        imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

        # Use clone().detach() to copy the tensor correctly
        imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            learning_rate=0.01
            max_iterations=5000
            #print('learning_rate ',learning_rate)
            #print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)
        for step in range(max_iterations):

            if active_mask.sum() == 0:
              break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_oimgs = oimgs[active_mask]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_valid = valid[active_mask]
            #active_CONST = CONST[active_mask]
            active_modifier.retain_grad()

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            #loss1 = criterion(output, active_labs_tensor)
            loss2 = l2dist
            loss = (const * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            optimizer.zero_grad()


            # Check for improvement in loss for each sample
            improved_mask = loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            #active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100


            with torch.no_grad():
                result = newimg.clone()
                result[result<0.5] = 0.
                result[result>0.5] = 1.
                output2 = model(result)

            '''if compare(output, labs_tensor, confidence, targeted):
                print('output ',output)
                print('OK')'''

            if compare(output2, active_labs_tensor, confidence, targeted).sum():
                active_mask2 = compare(output2, active_labs_tensor, confidence, targeted)
                print(active_mask2)

                print('output2 ',output2[active_mask2])
                #print(modifier.grad.detach().sum())
                gradientnorm, scores, nimg = active_modifier.grad[active_mask2].detach(), output[active_mask2].detach(), newimg[active_mask2].detach()

                equal_count = active_oimgs[active_mask2].shape[-1] - torch.sum(torch.abs(active_oimgs[active_mask2] - nimg) < 1e-4,1)
                print("Fixed features:", torch.sum(1 - active_valid[active_mask2],1), "Equal count:", equal_count)

                #if torch.sum(valid) == 0:
                    #last_solution = oimgs
                    #break

                valid0 = active_valid[active_mask2]
                #print('valid0',valid0.sum())
                totalchange = (nimg - active_oimgs[active_mask2]).abs() * gradientnorm.abs()

                totalchange = totalchange.flatten(1)

                for b in range(totalchange.shape[0]):
                    for e in torch.argsort(totalchange[b]):
                        if valid0[b, e]:
                            valid0[b, e] = 0
                            if totalchange[b, e] > 0.01:
                                break

                #print('valid0',valid0.sum())

                '''for e in torch.argsort(totalchange):
                    if valid0[e]:
                        valid0[e] = 0
                        if totalchange[e] > 0.01 :
                            break'''

                active_valid[active_mask2] = valid0.view((active_oimgs[active_mask2].shape))
                valid[active_mask] = active_valid
                print("Now Fixed features:", torch.sum((1 - valid),1))

                #update active mask
                #active_mask[active_mask][active_mask2] = False
                # Step 1: Identify the indices where active_mask is True
                true_indices = torch.nonzero(active_mask, as_tuple=True)[0]  # This gives the indices [0, 1, 2]

                # Step 2: Apply active_mask2 to modify these specific indices
                active_mask[true_indices[active_mask2]] = False
                #print('last avtive mask ',active_mask)

                #last_solution = prev = nimg
                last_solution[true_indices[active_mask2]] = nimg


        # Perform the AND NOT operation(which samples update)
        result = torch.logical_xor(active_mask,pre_active_mask)
        print(result.sum())
        if result.sum() != 0 :
            active_mask = result.clone().detach().to(dtype=torch.bool, device=device)

        else :
            i += 1
            const = CONST[i]
            break




if last_solution!=None:
    result = last_solution.clone()
    result[result<0.5] = 0.
    result[result>0.5] = 1.
    compare(result, labs, confidence, targeted)
    print('output = ',model(result))
    print('L0 = ',torch.sum(torch.abs(result - oimgs),1))
    print('------------------------------------')
else :
    print('Attack failed')
    print('------------------------------------')

In [ ]:
model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = False
learning_rate = 0.05
initial_const = 1e-3
largest_const = 2e9
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01

oimgs = mal_x_batch[1:2]
lab = mal_y_batch[1:2]

labs = torch.zeros_like(lab.squeeze(), dtype=torch.int64)

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

prev = oimgs.clone()
last_solution = None
const = initial_const

oimgs = oimgs.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# if model predict wrongly, no need to perturb
if compare(model(oimgs), labs_tensor, confidence, targeted):
    print('output = ',model(oimgs))
    print('L0 = ',torch.sum(torch.abs(oimgs - oimgs)))
    print('------------------------------------')


while const<largest_const:

    model.eval()

    CONST = [initial_const, 0.01,0.05,0.1,0.5,1.,10.,1000.,1e9,largest_const]
    i = CONST.index(const)

    #for const in CONST:
    while const < largest_const:
        print('CONST ',const)

        best_loss = torch.tensor(float('inf'), device=device)
        no_improvement_counter = 0  # Counter for no improvement

        # Calculate indices where mask is 0 or 1
        mask_0 = valid == 0
        mask_1 = valid == 1

        # Create empty tensor for the result
        imgs_tanh = torch.empty_like(oimgs)

        # Apply the transformations based on the mask
        imgs_tanh[mask_0] = torch_arctanh0(oimgs[mask_0])
        imgs_tanh[mask_1] = torch_arctanh1(oimgs[mask_1])

        # Use clone().detach() to copy the tensor correctly
        imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)

        modifier = torch.zeros_like(imgs_tensor, dtype=torch.float32, requires_grad=True, device=device)

        if const == 1e9:
            learning_rate=0.01
            max_iterations=5000
            #print('learning_rate ',learning_rate)
            #print('max_iterations ',max_iterations)

        optimizer = optim.Adam([modifier], lr=learning_rate)

        for step in range(max_iterations):

            optimizer.zero_grad()

            modified_input = imgs_tensor + modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            output = model(newimg)
            #l2dist = torch.sum((newimg - oimgs) ** 2, dim=1)
            l2dist = torch.sum((newimg - (torch.tanh(imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)

            # Calculate the loss based on predictions
            real = output[0, labs_tensor]
            other = output[0, (labs_tensor + (1 if targeted else -1))]


            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            loss2 = l2dist
            loss = (const * loss1) + loss2
            total_loss = torch.sum(loss)


            # Backpropagate the loss
            total_loss.backward()
            optimizer.step()

            #if step % ((max_iterations+1) // 50) == 0 :
                #print('output ',output)
                #print(modifier.grad.detach().sum())
                #print(compare(output, labs_tensor, confidence, targeted))

            # Check if the loss has improved
            if total_loss.item() < best_loss:
                best_loss = total_loss.item()
                no_improvement_counter = 0  # Reset the counter if improvement is found
            else:
                no_improvement_counter += 1  # Increment the counter if no improvement

            if abort_early and (no_improvement_counter >= 100):
                #print(f"No improvement for 100 consecutive iterations at step {step}. Breaking out of loop.")
                break

            #if abort_early and step % (max_iterations // 10) == 0 and total_loss.item() > prev_loss * .999999:
                #break

            #prev_loss = total_loss.item()

            with torch.no_grad():
                result = newimg.clone()
                result[result<0.5] = 0.
                result[result>0.5] = 1.
                output2 = model(result)

            '''if compare(output, labs_tensor, confidence, targeted):
                print('output ',output)
                print('OK')'''

            #print('step ', step)
            #print('output2 ', output2)
            if compare(output2, labs_tensor, confidence, targeted):
                print('*****************')

                print('output2 ',output2)
                #print(modifier.grad.detach().sum())
                gradientnorm, scores, nimg = modifier.grad.detach(), output.detach(), newimg.detach()
                equal_count = oimgs.shape[-1] - torch.sum(torch.abs(oimgs - nimg) < 1e-4).item()
                print("Fixed features:", torch.sum(1 - valid).item(), "Equal count:", equal_count)

                #if torch.sum(valid) == 0:
                    #last_solution = oimgs
                    #break

                valid = valid.flatten()
                totalchange = (nimg - oimgs).abs() * gradientnorm[0].abs()

                totalchange = totalchange.flatten()

                for e in torch.argsort(totalchange):
                    if valid[e]:
                        valid[e] = 0
                        if totalchange[e] > 0.01 :
                            break

                valid = valid.view((oimgs.shape))
                print("Now Fixed features:", torch.sum(1 - valid).item())
                last_solution = prev = nimg

                i -= 1
                break

        i += 1
        const = CONST[i]



if last_solution!=None:
    result = last_solution.clone()
    result[result<0.5] = 0.
    result[result>0.5] = 1.
    compare(result, labs, confidence, targeted)
    print('output = ',model(result))
    print('L0 = ',torch.sum(torch.abs(result - oimgs)))
    print('------------------------------------')
else :
    print('Attack failed')
    print('------------------------------------')

#New form of L0 attack

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

oimgs = mal_x_batch[1:2]
labs = target_label[1:2]
model = model_AT_rFGSM.to(device)
max_iterations = 1000
abort_early = True
learning_rate = 0.05
initial_const = 1e-3
largest_const = 1e3
REDUCE_CONST = False
targeted = True
const_factor = 10
confidence=0.01

# Ensure fixed_features_mask is a tensor
expanded_removal_array = removal_array.expand(oimgs.shape[0], -1)
valid = torch.bitwise_or(expanded_removal_array, 1 - oimgs.to(torch.uint8))
print('oimgs ', oimgs.sum(1))
print('fixed_features_mask ',(1 - valid).sum(1))

prev = oimgs.clone()
last_solution = None
const = initial_const

while True:
    res = gradient_descent(model, targeted, learning_rate, max_iterations, abort_early, const, largest_const,
                          const_factor, oimgs, labs, valid, device)
    if res is None:
        print("Final answer")
        break

    gradientnorm, scores, nimg, const = res
    if REDUCE_CONST:
        const /= 2.

    equal_count = oimgs.shape[-1] - torch.sum(torch.abs(oimgs - nimg) < 1e-4).item()
    print("Fixed features:", torch.sum(1 - valid).item(), "Equal count:", equal_count)

    if torch.sum(valid) == 0:
        last_solution = [oimgs]
        break


    valid = valid.flatten()
    totalchange = (nimg - oimgs) * gradientnorm[0]

    totalchange = totalchange.flatten()

    e = torch.argmax(totalchange)
    if ~valid[e]:
        valid[e] = 0

        if totalchange[e] < 0.001 :
            break
        else:
          print(gradientnorm[0,e])
          print('oimgs[0,e]',oimgs[0,e])
          if gradientnorm[0,e] > 0.:
              oimgs[0,e] = 1.
          else :
              oimgs[0,e] = 0.



    if compare(model(oimgs), labs, confidence, targeted):
      break

    valid = valid.view((oimgs.shape))
    print("Now Fixed features:", torch.sum(1 - valid).item())

    last_solution = prev = nimg

## Part 4 — CW examples for adversarial training

### Step 7 — CW-L2 without rounding, as a training-example generator

# CW L2 attack (no rounded)

In [ ]:
import torch
import torch.optim as optim
import numpy as np
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

print(mal_x_batch.shape)
print(mal_y_batch.shape)

target_label = torch.zeros_like(mal_y_batch.squeeze(), dtype=torch.int64)

def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

model = model_AT_rFGSM.to(device)
imgs = mal_x_batch
labs = target_label
targeted = True
#abort_early = True
confidence = 0.01
initial_const = 0.1
learning_rate = 0.1
binary_search_steps = 3
max_iterations = 1000
removal_array = removal_array

batch_size = imgs.shape[0]

expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

# Calculate indices where mask is 0 or 1
mask_0 = non_fixed_features_mask == 0
mask_1 = non_fixed_features_mask == 1

# Create empty tensor for the result
imgs_tanh = torch.empty_like(imgs)

# Apply the transformations based on the mask
imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

# Use clone().detach() to copy the tensor correctly
imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

# Set the lower and upper bounds for binary search
lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

# Variables to store the best results
o_bestl2 = torch.full((batch_size,), 1e10, device=device)
o_bestscore = torch.full((batch_size,), -1, device=device)
o_bestattack = imgs.clone().detach()
# o_best_loss1: for getting best loss1 for those samples with no successful attack
o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
o_best_unsuccessfull_attack = imgs.clone().detach()

# Set model to evaluation mode
model.eval()
#criterion = nn.CrossEntropyLoss(reduction='none')

for outer_step in range(binary_search_steps):

    if outer_step == (binary_search_steps - 1):
        max_iterations = 3000
        #learning_rate = 0.005

    print(f"Binary search step {outer_step}, max_iterations: {max_iterations}")

    # Initialize modifier with requires_grad=True
    modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

    # Setup optimizer
    optimizer = optim.Adam([modifier], lr=learning_rate)

    # Setup learning rate scheduler
    #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

    # To track the best loss for each sample and count of iterations without improvement
    best_loss = torch.full((batch_size,), float('inf'), device=device)
    no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

    # Track active samples
    active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

    bestl2 = torch.full((batch_size,), 1e10, device=device)
    bestscore = torch.full((batch_size,), -1, device=device)

    for iteration in range(max_iterations):
        if active_mask.sum() == 0:
            print(f"All samples are inactive at iteration {iteration}.")
            break

        # Filter active samples
        active_indices = torch.where(active_mask)[0]
        active_modifier = modifier[active_mask]
        active_imgs_tensor = imgs_tensor[active_mask]
        active_labs_tensor = labs_tensor[active_mask]
        active_CONST = CONST[active_mask]

        optimizer.zero_grad()

        # Compute new images in original space
        modified_input = active_imgs_tensor + active_modifier
        newimg = torch.tanh(modified_input) * 0.5 + 0.5

        # Get model predictions (pre-softmax)
        output = model(newimg.to(device))

        # Compute L2 distance to original images
        l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
        #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


        # Calculate the loss based on predictions
        real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
        other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

        # Calculate adversarial loss
        loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
        #loss1 = criterion(output, active_labs_tensor)
        loss2 = l2dist
        loss = (active_CONST * loss1) + loss2
        total_loss = torch.sum(loss)

        # Backpropagate the loss
        total_loss.backward()

        optimizer.step()

        # Step the learning rate scheduler
        #scheduler.step()

        with torch.no_grad():

            # Check for improvement in loss for each sample
            improved_mask = loss < best_loss[active_mask]
            best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
            no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

            # Check for improvement in loss1 for each sample
            improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 4.)
            o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
            o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

            # Update the best result found so far
            update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
            bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

            update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
            o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
            o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
            o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

            # Skip the rest of the loop for samples that haven't improved in 50 iterations
            active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100
            #active_mask[active_mask.clone()] = l2dist < 5.


            # Print losses every 10% of max_iterations
            #if iteration % (max_iterations // 20) == 0:
            '''if (iteration+1) % 200 == 0:
                #print('masks ',mask_one.sum(1),mask_zero.sum(1))
                print(output)
                #print('active_mask ', active_mask.sum())
                print(f"Iteration {iteration+1}, Total Loss: {total_loss.item()}")'''

    print(f"Binary search step {outer_step}, CONST: {CONST}")
    print(f"Binary search step {outer_step}, o_bestl2: {o_bestl2}")
    print(f"Binary search step {outer_step}, output: {model(o_bestattack)}")

    if outer_step != (binary_search_steps - 1) :
        # Create masks for successful and unsuccessful attacks
        success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
        failure_mask = ~success_mask

        # Update upper bounds and constants where attacks are successful
        upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

        # Update lower bounds and constants where attacks are unsuccessful
        lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

        # Handle cases where upper bound is still too high
        upper_limit_mask = upper_bound < 1e9
        CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
        CONST[failure_mask & ~upper_limit_mask] *= 2



# Final output based on the mask condition
# our goal is to perturb all samples even those without successful attack
success_attack_mask = (o_bestl2 != 1e10)
final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

In [ ]:
model_AT_rFGSM(final_output)

In [ ]:
((final_output - mal_x_batch)** 2).sum(1)

In [ ]:
(torch.abs(final_output - mal_x_batch)).sum(1)

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    targeted = True
    batch_size = imgs.shape[0]

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # o_best_loss1: for getting best loss1 for those samples with no successful attack
    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()

    for outer_step in range(binary_search_steps):

        '''if outer_step == (binary_search_steps - 1):
            max_iterations = 3000
            learning_rate = 0.01'''

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=learning_rate)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            #loss1 = criterion(output, active_labs_tensor)
            loss2 = l2dist
            loss = (active_CONST * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            with torch.no_grad():
                # Check for improvement in loss for each sample
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

                # Check for improvement in loss1 for each sample
                improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 4.)
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                # Update the best result found so far
                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1) :
          # Create masks for successful and unsuccessful attacks
          success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
          failure_mask = ~success_mask

          # Update upper bounds and constants where attacks are successful
          upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

          # Update lower bounds and constants where attacks are unsuccessful
          lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

          # Handle cases where upper bound is still too high
          upper_limit_mask = upper_bound < 1e9
          CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
          CONST[failure_mask & ~upper_limit_mask] *= 2

    # Final output based on the mask condition
    # our goal is to perturb all samples even those without successful attack
    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
with torch.no_grad():
    for x_test, y_test in test_loader:
        x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
        mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
        break

In [ ]:
final_output = CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            print((y_pred == 0))
            print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
#CW(mal_x_batch, mal_y_batch, model_AT_rFGSM, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, device=device)
attack_params =  {'device':device, 'confidence':0.01, 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 0.01, 'initial_const': 0.1, 'binary_search_steps' : 3}
adv_predict(test_loader, model_AT_rFGSM, CW, **attack_params)

### Step 8 — Adversarial training of the SupCon model with CW in the loop

# SupContrast Model (Encoder + Projector)

In [ ]:
import torch
import torch.nn.functional as F

def calculate_divergences(z, labels):
    # Ensure the tensors are on the same device
    device = z.device
    labels = labels.to(device)

    # Compute cosine similarities
    cos_sim_z = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

    # Convert cosine similarities to cosine distances
    cos_distances_z = 1 - cos_sim_z

    # Initialize variables
    N = len(z)
    intra_class_distances = []
    inter_class_distances = []

    for i in range(N):
        # Identify positive and negative sets for the anchor
        positive_indices = (labels == labels[i]).nonzero(as_tuple=True)[0]
        negative_indices = (labels != labels[i]).nonzero(as_tuple=True)[0]

        # Compute intra-class divergence for benign and adversarial examples
        intra_class_distance_z = cos_distances_z[i][positive_indices].mean().item()

        # Compute inter-class divergence for benign and adversarial examples
        inter_class_distance_z = cos_distances_z[i][negative_indices].mean().item()

        intra_class_distances.append(intra_class_distance_z)
        inter_class_distances.append(inter_class_distance_z)

    # Calculate absolute intra-class and inter-class divergences
    d_a_plus = torch.tensor(intra_class_distances).mean().item()
    d_a_minus = torch.tensor(inter_class_distances).mean().item()

    # Calculate relative intra-class divergence (R-DIV)
    d_r_plus = d_a_plus / d_a_minus

    print("Absolute Intra-Class Divergence:", d_a_plus)
    print("Absolute Inter-Class Divergence:", d_a_minus)
    print("Relative Intra-Class Divergence (R-DIV):", d_r_plus)


    return d_a_plus, d_a_minus, d_r_plus

# Example usage
batch_size = 100
latent_dim = 200
num_classes = 2

# Generate random latent representations and labels for demonstration
z = torch.rand(batch_size, latent_dim)
labels = torch.randint(0, num_classes, (batch_size,))

# Calculate divergences
d_a_plus, d_a_minus, d_r_plus = calculate_divergences(z, labels)

In [ ]:
def extract_features(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            output, z = model.CoreModel(inputs.to(torch.float32), return_z=True)
            features.append(z.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels


def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title('t-SNE Scatter Plot')

    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    ax2.set_title('Density Plot')

    plt.show()

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

In [ ]:
def pgd_min22(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        #loss = torch.clamp((real - other + 0.1), min=0.0)
        loss = (real - other)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh(((x - 0.5) / 0.5) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, cons_increase_factor=2., device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    targeted = True
    batch_size = imgs.shape[0]

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # o_best_loss1: for getting best loss1 for those samples with no successful attack
    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()

    for outer_step in range(binary_search_steps):

        '''if outer_step == (binary_search_steps - 1):
            max_iterations = 3000
            learning_rate = 0.01'''

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=learning_rate)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if active_mask.sum() == 0:
                #print(f"All samples are inactive at iteration {iteration}.")
                break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = torch.tanh(modified_input) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            #l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence) if targeted else (real - other + confidence), min=0.0)
            #loss1 = criterion(output, active_labs_tensor)
            loss2 = l2dist
            loss = (active_CONST * loss1) + loss2
            total_loss = torch.sum(loss)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            with torch.no_grad():
                # Check for improvement in loss for each sample
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

                # Check for improvement in loss1 for each sample
                improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 4.)
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                # Update the best result found so far
                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 100

        if outer_step != (binary_search_steps - 1) :
          # Create masks for successful and unsuccessful attacks
          success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
          failure_mask = ~success_mask

          # Update upper bounds and constants where attacks are successful
          upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

          # Update lower bounds and constants where attacks are unsuccessful
          lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

          # Handle cases where upper bound is still too high
          upper_limit_mask = upper_bound < 1e9
          CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / 2
          CONST[failure_mask & ~upper_limit_mask] *= cons_increase_factor

    # Final output based on the mask condition
    # our goal is to perturb all samples even those without successful attack
    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
import torch
import torch.nn.functional as F

def distance(x, y, dist, pairwise=False):
    """
        Pairwise distance
        Args:
            x, y: input pair
            dist: distance type ["l2", "l1", "linf", "cosine"]
            pairwise:
        Note:
            Need renormalize if using l1
    """

    assert(len(x.shape) == 2)
    b, d = x.shape

    if pairwise:
        x = torch.unsqueeze(x, dim=0) # [1, b, d]
        y = torch.unsqueeze(y, dim=1) # [b, 1, d]

    if dist == 'l2':
        loss = torch.sqrt(torch.sum(torch.square(x - y), dim=-1, keepdim=False)) # [b,b]

    elif dist == 'l1':
        loss = torch.sum(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'linf':
        loss, _ = torch.max(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'lkinf':
        k = 10
        t, _ = torch.topk(torch.abs(x - y), k=k, dim=-1)
        loss = torch.sum(t, dim=-1, keepdim=False)
        loss /= k

    elif dist == 'cosine':
        # Note that: cosine_similarity range [-1,1]
        # dist=-1 <--> very similar
        # dist=0 <--> orthogonal
        # dist=1 <--> very disimilar
        loss = 0 - torch.cosine_similarity(x, y, dim=-1)

    elif dist == 'matmul':
        x = torch.squeeze(x) # [b,d]
        y = torch.squeeze(y) # [b,d]
        loss = - torch.matmul(x, y.T)

    # loss = loss / torch.max(loss).detach() # normalizing to avoid numerical problem

    if pairwise:
        assert(loss.shape[0] == b)
        assert(loss.shape[1] == b)
    else:
        assert(loss.shape[0] == x.shape[0]) # [b, ]

    return loss

def get_positive_mask(labels):
    # ATTENTION HERE, positive mask will ignore the diagonal

    # l = torch.matmul(labels, labels.T) # [2b,2b]
    # m = torch.ones_like(l).fill_diagonal_(0) # [2b,2b]
    # pos_mask = torch.multiply(l, m) # [2b,2b]

    l = torch.matmul(labels, labels.T).fill_diagonal_(0) # [2b,2b]
    return l

def get_negative_mask(labels):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    return inv_mask

def get_hard_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    hard_mask = torch.matmul(labels, preds.T) # [2b, 2b], mask for tile(y_j) == y_i
    neg_mask = torch.multiply(inv_mask, hard_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask


def get_soft_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    soft_mask = torch.matmul(preds, preds.T) # [2b, 2b], mask for tile(y_j) == tile(y_i)
    neg_mask = torch.multiply(inv_mask, soft_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask

LARGE_NUM = 1e9

def soft_lcscl(hidden,
             labels,
             preds,
             dist,
             neg_type,
             hidden_norm=True,
             temperature=0.07):
    """Compute Local Supervised Contrastive Loss.
    Args:
        hidden: hidden vector (`Tensor`) of shape (2 * bsz, dim).
        labels: labels of hidden vector of shape (2*bsz, classes)
        preds: current prediction
        hidden_norm: whether or not to use normalization on the hidden vector.
        temperature: a `floating` number for temperature scaling.
        dist: distance metric in latent space {'l1', 'l2', 'cosine', 'linf'}
        neg_type:
            'hard': negative set when y_j ~= y_i, but tilde(y_j) == y_i
            'soft': negative set when y_j ~= y_i, but tilde(y_j) == tilde(y_i)
            'merge': negative set when y_i ~= y_i but tilde(y_j) in [y_i,  tilde(y_i)]
            'leaking':
    Returns:
        A loss scalar.

    the choice of temperature:
        temperature = 0.5 based on simCLR paper
        temperature = 0.07 based on SCL paper (they claim that smaller temperature
        benefit training more than higher ones)
    """

    # Get (normalized) hidden1 and hidden2.
    if hidden_norm:
        # gradient will pass through the normalization step
        hidden = F.normalize(hidden, p=2, dim=-1)


    batch_size = hidden.shape[0]

    # Remove illegal elements
    same_preds = get_positive_mask(preds)
    same_labels = get_positive_mask(labels)
    diff_labels = get_negative_mask(labels)

    same_preds_same_labels = torch.multiply(same_preds, same_labels)
    same_preds_diff_labels = torch.multiply(same_preds, diff_labels)

    pos_mask = get_positive_mask(labels)
    # neg_mask = tf.ones_like(pos_mask) - pos_mask # INCORRECT, because it consider the diagonal as NEGATIVE
    neg_mask = get_negative_mask(labels) # CORRECT
    hard_neg_mask = get_hard_negative_mask(labels, preds)
    soft_neg_mask = get_soft_negative_mask(labels, preds)
    merge_mask = torch.minimum(hard_neg_mask+soft_neg_mask, torch.ones_like(hard_neg_mask))

    # ATTENTION HERE, NEED VERIFIED BY EXPERIMENT, Stop_gradient make it worse
    # pos_mask = pos_mask.detach()
    # neg_mask = neg_mask.detach()
    # hard_neg_mask = hard_neg_mask.detach()
    # soft_neg_mask = soft_neg_mask.detach()
    # merge_mask = merge_mask.detach()
    # same_preds_same_labels = same_preds_same_labels.detach()
    #
    if neg_type == 'all':
        illegal_mask = torch.zeros_like(neg_mask)
        nb_neg = torch.sum(neg_mask, dim=-1)

    elif neg_type == 'hard':
        illegal_mask = torch.maximum(neg_mask - hard_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(hard_neg_mask, dim=-1)

    elif neg_type == 'soft':
        illegal_mask = torch.maximum(neg_mask - soft_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(soft_neg_mask, dim=-1)

    elif neg_type == 'merge':
        illegal_mask = torch.maximum(neg_mask - merge_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(merge_mask, dim=-1)

    elif neg_type == 'leaking':
        illegal_mask = torch.maximum(torch.ones_like(same_preds_same_labels) - same_preds_same_labels - same_preds_diff_labels, torch.zeros_like(same_preds_same_labels))
        #print(illegal_mask)
        nb_neg = torch.sum(same_preds_diff_labels, dim=-1)

    # Pairwise Similarity
    masks = F.one_hot(torch.arange(batch_size), num_classes=batch_size).to(hidden.device)
    sim = - distance(hidden, hidden, dist, pairwise=True) / temperature
    logits_12 = sim - masks * LARGE_NUM
    logits_masked = logits_12 - LARGE_NUM * illegal_mask
    logprobs = F.log_softmax(logits_masked, dim=1)

    """ Convert labels to pairwise labels of shape [2b, 2b], diagonal(labels)=0
        multiple hot label vectors

    """
    # Cross entropy loss with multiple hot labels

    if neg_type == 'leaking':
        loss = - torch.sum(same_preds_same_labels * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(same_preds_same_labels, dim=-1), torch.ones_like(loss)) # [2b, ]
    else:
        loss = - torch.sum(pos_mask * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(pos_mask, dim=-1), torch.ones_like(loss)) # [2b, ]

    # if True:
    #     scale = torch.div(nb_neg, nb_neg+nb_pos) # V1
    #     # scale = torch.div(nb_neg, nb_pos) #V3
    #     # scale = nb_neg #V2
    #     loss = torch.multiply(loss, scale)

    assert(loss.shape == nb_pos.shape)
    #print(nb_pos)
    loss = torch.div(loss, nb_pos) # [2b,]
    loss = torch.mean(loss, dim=0) # [1,]

    return loss

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

def adversarial_training(model, train_loader, val_loader, attack, epochs=100, lr=0.001, weight_decay=0., beta=1., device='cuda', verbose=True, **kwargs):

    # Define parameters for the loss function
    dist = 'matmul'
    neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
    hidden_norm = False
    temperature = 0.07


    # Define Loss Function and Optimizer
    #class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion_at = nn.CrossEntropyLoss()
    #optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.1, 0.999), weight_decay=weight_decay)
    #momentum = 5e-4
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)


    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(epochs):
        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0
        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])

            # bsz must be updated due to added adversarial samples
            bsz = x_batch.shape[0]

            model.train()
            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            #print(f'Training loss (batch level): {loss_train.item():.4f} | Train accuracy: {((predicted == y_batch.view(-1)).sum().item()/bsz):.2f}%')

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz


        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        #if (pre_loss - mean_loss) < 1e-2:
            #target_model.load_state_dict(model.state_dict())
            # because after new adversarial samples loss increase so mostly after just one epoch this condition would be true
            # so we change pre_loss to inf for training more than one epoch
            #mean_loss = torch.inf


        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        if verbose:
            print(f"\tVal accuracy (without attack) {(avg_acc_val) * 100:.4}% and accuracy (with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel selected at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

def generate_adversarial_examples(data_loader, model, attack, device=device, **kwargs):
    mal_x_samples = []
    labels = []

    model.eval()
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)

        mal_x_batch, mal_y_batch = x_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 1]

        mal_x_samples.append(mal_x_batch)
        labels.append(mal_y_batch)

    # Concatenate all adversarial examples and labels
    mal_x_samples = torch.cat(mal_x_samples, dim=0)
    labels = torch.cat(labels, dim=0)

    adversarial_examples = attack(mal_x_samples, labels, model, **kwargs)

    return adversarial_examples, labels

def adversarial_training(model, train_loader, val_loader, attack, epochs=100, lr=0.001, weight_decay=0., beta=1., device=device, verbose=True, **kwargs):

    # Define parameters for the loss function
    dist = 'matmul'
    neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
    hidden_norm = False
    temperature = 0.07

    # Define Loss Function and Optimizer
    #class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion_at = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)

    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    # VALIDATION PART
    samples = []
    labels = []

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
        samples.append(x_batch)
        labels.append(y_batch)

    # Concatenate all adversarial examples and labels
    x_val = torch.cat(samples, dim=0)
    y_val = torch.cat(labels, dim=0)

    for epoch in range(epochs):
        model.eval()
        # Generate adversarial examples for the entire training dataset
        adv_examples, adv_labels = generate_adversarial_examples(train_loader, model, attack, device, **kwargs)

        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0

        # Iterate over the batches in the original data loader
        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):

            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz_raw = x_batch.shape[0]
            if bsz_raw < 100:
                continue

            mal_y_batch = y_batch[y_batch.squeeze() == 1]
            bsz_mal = mal_y_batch.shape[0]


            # Retrieve the corresponding adversarial examples for this batch
            adv_x_batch = adv_examples[idx_batch * bsz_mal: (idx_batch + 1) * bsz_mal]
            adv_y_batch = adv_labels[idx_batch * bsz_mal: (idx_batch + 1) * bsz_mal]

            x_batch = torch.cat([x_batch, adv_x_batch], dim=0)
            y_batch = torch.cat([y_batch, adv_y_batch])

            # bsz must be updated due to added adversarial samples
            bsz = x_batch.shape[0]

            model.train()
            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz

        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        # VALIDATION
        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_samples = len(x_val)
        outputs = model(x_val)
        predicted = outputs.argmax(1).unsqueeze(1)
        cor_val = (predicted == y_val).sum().item()

        mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
        n_ad_samples = len(mal_x_batch)
        attack_params =  {'confidence':0.01, 'removal_array':removal_array,'max_iterations': 1000, 'learning_rate': 0.01, 'initial_const': 1e4, 'binary_search_steps' : 1, 'cons_increase_factor' : 10}
        pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **attack_params)
        outputs = model(pertb_mal_x)
        y_pred = outputs.argmax(1)
        cor_ad_val = (y_pred == 1.).sum().item()

        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        if verbose:
            print(f"\tVal accuracy (without attack) {(avg_acc_val) * 100:.4}% and accuracy (with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel selected at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 10000
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

epochs = 400
attack_params =  {'confidence':0.01, 'removal_array':removal_array,'max_iterations': 500, 'learning_rate': 0.01, 'initial_const': 0.1, 'binary_search_steps' : 3, 'cons_increase_factor' : 2}
adversarial_training(combined_model.to(device), train_loader, val_loader, CW, epochs, lr=0.01, weight_decay=1e-4, beta=1., device=device, verbose=True, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(val_loader, combined_model, pgd_min22, device, **attack_params)

In [ ]:
combined_model2 = Wrapper(core_model, proj_head).to(device)
combined_model2.load_state_dict(torch.load('/content/model_AT_max.pth'))

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':100,'steps_max': 5, 'rounding':True, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2.to(device), PGD_Max2, device, **attack_params)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":100, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":100, 'is_report_loss_diff' : False}),
    (bca_k, {"k":100, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":100, 'is_report_loss_diff' : False}),
    #(mimicry, {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':1,'is_sample':False, 'varepsilon':1e-20}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':5,'is_sample':False, 'varepsilon':1e-20}),
    (StepwiseMax_onestep2, {'insertion_array':insertion_array, 'removal_array':removal_array,'step_lengths':{"l1": 1.0, "l2": 0.05, "linf": 0.0008}, "steps":650}),

]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model2.to(device), attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

In [ ]:
features, labels = extract_features(combined_model2, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":30, 'is_report_loss_diff':False}
adv_predict(test_loader, combined_model2.to(device), mimicry, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}
adv_predict(test_loader, combined_model2.to(device), mimicry, device, **attack_params)

In [ ]:
model_evaluation(combined_model2.to(device), test_loader, device)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)

## Part 5 — DDN in this workspace

### Step 9 — The evaluation harness and the DDN runs

In [ ]:
def pgd_min22(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', confidence=0.1, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    #x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        #loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))

        batch_size = x_var.shape[0]
        real = y_model[torch.arange(batch_size), y.view(-1).long()]
        other = y_model[torch.arange(batch_size), (y.view(-1).long()- 1)]
        loss = torch.clamp((real - other + confidence), min=0.0)
        #loss = (real - other)


        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *(x_var < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    #x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    #loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    #replace_flag = (loss_adv > loss_natural).squeeze()
    #x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 100000
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, n):
    """
    Calculate the sum of the top n smallest differences between adv_rounded and mal_x_batch
    for samples where y_pred == 0.

    Parameters:
    adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
    mal_x_batch (torch.Tensor): The original malicious examples tensor.
    y_pred (torch.Tensor): The prediction tensor.
    n (int): The number of top differences to sum.

    Returns:
    torch.Tensor: The sum of the top n smallest differences.
    """
    # Step 1: Calculate the total differences
    total_difference = torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0]).sum(1)

    # Step 2: Sort the differences in ascending order
    sorted_differences, _ = torch.sort(total_difference)

    # Step 3: Select the top n least differences
    top_n_least_differences = sorted_differences[:n]

    # Step 4: Sum these n differences
    sum_top_n_least_differences = top_n_least_differences.sum()

    return sum_top_n_least_differences

In [ ]:
def evaluate_non_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model):
    """
    Evaluate the model's performance on non-rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Get the model's predictions on adversarial examples
    outputs = combined_model(adv)
    y_pred = outputs.argmax(dim=1)

    # Calculate metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv, mal_x_batch, y_pred, 100)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on NOT_Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total difference (L1 norm) between successful adversarial examples and original: {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Accuracy of the model on malwares under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "accuracy": accuracy
    }

In [ ]:
def evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array):
    """
    Evaluate the model's performance on rounded adversarial examples.

    Parameters:
    - mal_x_batch: Original malware input batch
    - mal_y_batch: Ground truth labels for malware batch
    - adv: Adversarial examples generated from mal_x_batch
    - combined_model: The trained model being evaluated
    - removal_array: Array used in the final rounding step

    Returns:
    - A dictionary containing evaluation metrics
    """

    # Set the model to evaluation mode
    combined_model.eval()

    # Apply the final rounding solution to adversarial examples
    adv_rounded = final_rounding(mal_x_batch, adv, combined_model, removal_array)

    # Get the model's predictions on the rounded adversarial examples
    outputs = combined_model(adv_rounded)
    y_pred = outputs.argmax(dim=1)

    # Calculate evaluation metrics
    num_malwares = len(mal_y_batch)
    num_successful_attacks = (y_pred == 0).sum().item()
    total_difference = (torch.abs(adv_rounded[y_pred == 0] - mal_x_batch[y_pred == 0])).sum().item()
    total_difference_TOP100 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 100)
    total_difference_TOP150 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 150)
    total_difference_TOP200 = sum_of_top_n_differences(adv_rounded, mal_x_batch, y_pred, 200)
    accuracy = (y_pred == 1).sum().item() / num_malwares * 100

    # Print the evaluation results
    print(f"Evaluation Results on Rounded Adversarial Examples")
    print(f"-----------------------------------------------")
    print(f"Number of malware samples: {num_malwares}")
    print(f"Number of successful attacks: {num_successful_attacks}")
    print(f"Total L1 norm difference (successful adversarial examples vs. original): {total_difference}")
    print(f"Total L1 norm difference - TOP 100 (successful adversarial examples vs. original): {total_difference_TOP100}")
    print(f"Total L1 norm difference - TOP 150 (successful adversarial examples vs. original): {total_difference_TOP150}")
    print(f"Total L1 norm difference - TOP 200 (successful adversarial examples vs. original): {total_difference_TOP200}")
    print(f"Accuracy of the model on malware under attack: {accuracy:.2f}%")

    # Return results as a dictionary
    return {
        "adv_rounded": adv_rounded,
        "num_malwares": num_malwares,
        "num_successful_attacks": num_successful_attacks,
        "total_difference": total_difference,
        "total_difference_TOP100": total_difference_TOP100,
        "total_difference_TOP150": total_difference_TOP150,
        "total_difference_TOP200": total_difference_TOP200,
        "accuracy": accuracy
    }

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, adv_rounded, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - adv_rounded: Rounded adversarial examples
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()
    diff_adv_rounded_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_rounded_non_fixed = (torch.abs(adv_rounded - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_fixed}")
    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")
    print(f"  Rounded Adv vs. Original: {diff_adv_rounded_non_fixed}")

    # Return the results as a dictionary
    return {
        "diff_adv_fixed": diff_adv_fixed,
        "diff_adv_non_fixed": diff_adv_non_fixed,
        "diff_adv_rounded_fixed": diff_adv_rounded_fixed,
        "diff_adv_rounded_non_fixed": diff_adv_rounded_non_fixed
    }

In [ ]:
def compare_round(mal_x_batch,newimg, model,removal_array):

    adv_rounded = final_rounding(mal_x_batch, newimg, model,removal_array)
    #l0dist = torch.sum((mal_x_batch - newimg)!=0, dim=1)
    l0dist = torch.sum(torch.abs(adv_rounded - mal_x_batch), dim=1)
    logits = model(adv_rounded)
    output = torch.argmax(logits, dim=1)

    return output == 0, l0dist

In [ ]:
def adv_predict_rounding(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            adv_rounded = final_rounding(mal_x_batch, pertb_mal_x, model,removal_array)

            outputs = model(adv_rounded)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

            #print(outputs)
            print(len(mal_y_batch))
            print((y_pred == 0).sum().item())
            #print((y_pred == 0))
            #print('mean difference ',(torch.abs(pertb_mal_x - mal_x_batch)).sum()/len(mal_y_batch))
            #print((torch.abs(pertb_mal_x[y_pred == 0] - mal_x_batch[y_pred == 0])).sum())
            #print((torch.abs(pertb_mal_x - mal_x_batch)).sum(1))

            #print('***************************')


    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

    return pertb_mal_x

In [ ]:
def final_rounding(oimgs, newimg, model, removal_array):
    # Expand removal array to match the batch size
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).to(device)

    # Ensure all tensors are on the correct device
    oimgs, perturbed = oimgs.to(device), newimg.to(device)

    # Set the model to evaluation mode
    model.eval()

    # Calculate predictions and significant changes efficiently using GPU
    with torch.no_grad():
        predictions = model(oimgs).argmax(dim=1).to(device)
        active_mask = predictions == 1  # Efficiently create active mask on GPU
        total_change = (perturbed - oimgs).abs().to(device)  # Calculate absolute difference on GPU
        significant_mask = total_change > 0.1  # Determine significant changes on GPU
        sorted_idx = total_change.argsort(dim=1, descending=True)  # Sort on GPU
        changes = (perturbed > oimgs).float().to(device)

    # Copy original images to apply final rounding
    final = oimgs.clone().to(device)

    # Iterate up to the maximum number of significant changes
    # practicaly we dont have perturbs more than 120 !!
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 120)

    for i in range(max_significant_changes):
        # Safeguard against empty active mask
        if not active_mask.any():
            break

        # Get the indices of active samples
        active_indices = torch.where(active_mask)[0].to(device)

        # Get the relevant indices for current change
        indices = sorted_idx[active_indices, i]

        # since max_significant_changes is relatively large for some samples, possible that zero changes which belong to fixed_features get updated
        final[active_indices, indices] = torch.where(non_fixed_features_mask[active_indices, indices],
                                                     changes[active_indices, indices], final[active_indices, indices])

        # Update the active_mask based on the model's prediction
        with torch.no_grad():
            active_mask[active_indices] = (model(final[active_indices]).argmax(dim=1) == 1)

    return final

In [ ]:
for x_test, y_test in test_loader:
    x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
    mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
    ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
    print('mal_x_batch ',mal_x_batch.shape)
    print('ben_x_batch ',ben_x_batch.shape)
    break

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def plot_adv_distances(adv_rounded_list, labels, model, mal_x_batch2, device):
    """
    Plots the cumulative distance between adversarial tensors and a reference batch (mal_x_batch2).

    Parameters:
    - adv_rounded_list: List of adversarial tensors to be evaluated.
    - labels: List of labels for the adversarial tensors (for the plot legend).
    - model: The model used for predicting adversarial examples.
    - mal_x_batch2: The reference batch to compare adversarial tensors against.
    - device: Device to run the computations on (e.g., 'cuda' or 'cpu').
    - model_AT_rFGSM: A variant of the model to evaluate adversarial tensors.
    """

    # Set a style using Seaborn and select a color palette automatically
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))  # Automatically select colors

    # Create the plot
    plt.figure(figsize=(20, 12))

    model.eval()

    # Iterate through each adv_rounded tensor and plot their distances
    for adv_rounded, color, label in tqdm(zip(adv_rounded_list, colors, labels),
                                          total=len(adv_rounded_list), desc="Processing Tensors"):
        # Pass the adv_rounded tensor through the model to get predictions
        adv_rounded = adv_rounded.to(device)  # Move tensor to the correct device
        outputs = model(adv_rounded)
        y_pred = outputs.argmax(dim=1)

        # Calculate the differences between adv_rounded and mal_x_batch2 for correctly classified samples
        mal_x_batch2 = mal_x_batch2.to(device)
        total_difference = torch.sum((adv_rounded[y_pred == 0] - mal_x_batch2[y_pred == 0]) ** 2, dim=1)

        # Sort the differences in ascending order
        sorted_differences, _ = torch.sort(total_difference)

        # Generate the x-axis (distance values) and y-axis (cumulative counts)
        x_values = sorted_differences.cpu()  # Move tensor back to CPU for plotting
        y_values = torch.arange(1, len(sorted_differences) + 1).cpu()  # Create a tensor for the y-axis values

        # Plot the line for the current adv_rounded tensor
        sns.lineplot(x=x_values, y=y_values, marker='o', color=color, linewidth=2, markersize=5, label=label)

    # Add labels and title with a bit more formatting
    plt.xlabel('Distance', fontsize=16, labelpad=10)
    plt.ylabel('Number of Valid Samples (Cumulative)', fontsize=16, labelpad=10)
    plt.title('Cumulative Count of Valid Samples vs. Distance', fontsize=18, pad=15)

    # Add legend for clarity
    plt.legend(title='Adversarial Tensors', fontsize=12, title_fontsize=14, loc='upper left')

    # Improve the look of the grid and remove spines for a cleaner look
    plt.grid(True, linestyle='--', linewidth=0.5)
    sns.despine()

    # Show the plot with tighter layout for better visualization
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import numpy as np

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t / (2 * T)))

def compare(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def PGD(x, model, removal_array, max_iterations, min_lr, max_lr, gamma=0.05, device='cpu'):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we aim to minimize).
    :param x: Input feature vector.
    :param model: Neural network model.
    :param removal_array: Array indicating removable features.
    :param max_iterations: Maximum number of iterations for the attack.
    :param min_lr: Minimum learning rate for cosine annealing.
    :param max_lr: Maximum learning rate for cosine annealing.
    :param confidence: Confidence margin for attack.
    :param gamma: Factor for adaptive norm adjustment.
    :param device: Device for computation (CPU or GPU).
    :return: Adversarially perturbed version of x.
    """
    model.eval()
    model.to(device)

    # Initialize variables
    modifier = torch.zeros_like(x, requires_grad=True).to(device)
    optimizer = optim.SGD([modifier], lr=0.01)
    scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iterations, eta_min=min_lr)

    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_best_modifier = torch.zeros_like(x)
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    norm = torch.ones(batch_size, device=device)

    # Expand removal array and prepare the non-fixed features mask
    removal_array = removal_array.expand(x.shape[0], -1).to(device)
    non_fixed_features_mask = torch.bitwise_or(removal_array, 1 - x.to(torch.uint8)).to(device)

    for t in range(max_iterations):
        scheduler.step()
        lr = scheduler.get_last_lr()[0]

        # Forward pass with adversarial example
        x_adv = x + modifier
        y_model = model(x_adv)
        real = y_model[:, 0]
        other = y_model[:, 1]
        loss = (other - real)
        total_loss = loss.sum()

        # Calculate success for the current batch
        with torch.no_grad():
            l2dist = torch.norm(modifier, p=2, dim=1)
            success_mask_inloop = compare(y_model)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
            o_best_modifier = torch.where(update_mask2.unsqueeze(-1), modifier.data, o_best_modifier)

        # Backpropagation and gradient normalization
        optimizer.zero_grad()
        total_loss.backward()

        # Apply the non-fixed features mask
        modifier.grad.data *= non_fixed_features_mask

        # Normalize gradients to prevent NaN issues
        grad_norms = modifier.grad.data.norm(p=2, dim=1, keepdim=True).clamp_min(1e-6)
        modifier.grad.data /= grad_norms

        # Gradient step
        optimizer.step()

        # Update norms using the DDN-like adaptive method
        norm *= (1 - (2 * success_mask_inloop.float() - 1) * gamma)
        print(norm[:8])
        print(norm.shape)

        # Apply renormalization to the modifier and clamp to the valid range
        modifier.data = modifier.data * (norm.view(-1, 1) / modifier.data.norm(2, dim=1, keepdim=True).clamp_min(1e-6))

        modifier.data.add_(x)
        modifier.data.clamp_(0, 1).sub_(x)

        # Print progress every few iterations for debugging
        if t == 0 or (t + 1) % 1 == 0:
            print(f"Iteration {t + 1:5}:\t Current LR = {lr:.4f}\t Current success = {success_mask_inloop.sum().item()}\t success = {success_mask.sum().item()}\t Dist = {o_bestl2[success_mask].mean().item():.2f}\t Loss = {loss.mean().item():.2f}")

    return x + o_best_modifier

In [ ]:
adv = PGD(mal_x_batch, model_AT_rFGSM, removal_array, 200, 1e-3, 1, gamma=0.02, device='cpu')

In [ ]:
adv = PGD(mal_x_batch, model_AT_rFGSM, removal_array, 200, 1e-3, 1, gamma=0.01, device='cpu')

In [ ]:
import torch.optim.lr_scheduler as lr_scheduler
from torch.optim.lr_scheduler import CyclicLR
import math

def cyclical_lr(step, step_size_up, step_size_down, min_lr, max_lr):
    cycle_length = step_size_up + step_size_down
    cycle_position = step % cycle_length

    if cycle_position < step_size_up:
        lr = min_lr + (max_lr - min_lr) * (cycle_position / step_size_up)
    else:
        lr = max_lr - (max_lr - min_lr) * ((cycle_position - step_size_up) / step_size_down)

    return lr

def lr_cosine_half(t, T, lr_start, lr_end):
    return lr_start + (lr_end - lr_start) * (1 - np.cos(np.pi * t /(2*T)))

def compare2(output):
    output = torch.argmax(output, dim=1)
    return output == 0

def compare(outputs, confidence):
    outputs[:,0] -= confidence
    output = torch.argmax(outputs, dim=1)
    return output == 0

def PGD(x, model, removal_array, max_iterations, min_lr, max_lr, step_size_up=2000,step_size_down=2000, confidence=0.1, device=device):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()
    model.to(device)
    x.to(device)

    # Initialize starting point
    modifier = torch.zeros_like(x, requires_grad=True).to(device)

    # Expand insertion_array and removal_array to match the batch size
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8)).to(device)

    batch_size = x.shape[0]
    success_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_best_modifier = torch.zeros_like(x, device=device)
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    norm = torch.ones(batch_size, device=device)

    for t in range(max_iterations):

        # Forward pass with adversarial example
        x_adv = x + modifier
        y_model = model(x_adv)
        real = y_model[:, 0]
        other = y_model[:, 1]
        loss = torch.clamp((other - real + confidence), min=0.0)
        total_loss = loss


        with torch.no_grad():
            #l2dist = torch.norm(modifier, p=2, dim=1)
            l2dist =torch.sum((modifier) ** 2, dim=1)
            success_mask_inloop = compare(y_model, confidence)
            success_mask = torch.logical_or(success_mask_inloop, success_mask)

            update_mask2 = (l2dist < o_bestl2) & success_mask_inloop
            o_bestl2 = torch.where(update_mask2, l2dist, o_bestl2)
            o_best_modifier = torch.where(update_mask2.unsqueeze(-1), modifier.data, o_best_modifier)

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), modifier)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * ((x+modifier) > 0.) * removal_array_updated * gradients

        grad4insertion = (gradients >= 0) *((x+modifier) < 1.)* gradients
        #grad4removal = (gradients < 0) * (x_var > 0.) * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
        perturbation = (gradients / (l2norm + 1e-20)).float()



        # Update x_next
        #step_length = cyclical_lr(t, step_size_up, step_size_down, min_lr, max_lr)
        step_length = lr_cosine_half(t, max_iterations, min_lr, max_lr)
        modifier.data = modifier.data + step_length * perturbation
        modifier.data.add_(x)
        modifier.data.clamp_(0, 1).sub_(x)

        if t == 0 or (t + 1) % (max_iterations // 50) == 0:
          print(f"Iteration {t + 1:5}:\t Current LR = {step_length:.4f}\t success = {success_mask.sum().item()}\t Dist = {o_bestl2[success_mask].mean().item():.2f}\t Loss = {loss.mean().item():.2f}")

    return x + o_best_modifier

In [ ]:
adv = PGD(mal_x_batch, model_AT_rFGSM, removal_array, 20000, 0.001, 5., step_size_up=15000,step_size_down=5000, confidence=0.4, device=device)

In [ ]:
results_rounded = evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, model_AT_rFGSM, removal_array)

In [ ]:
plot_adv_distances([results_rounded['adv_rounded'],adv], ['adv_rounded','adv'], model_AT_rFGSM, mal_x_batch, device)

In [ ]:
adv = PGD(mal_x_batch, combined_model, removal_array, 20000, 0.001, 5., step_size_up=15000,step_size_down=5000, confidence=0.4, device=device)

In [ ]:
results_rounded = evaluate_rounded_adversarial_examples(mal_x_batch, mal_y_batch, adv, combined_model, removal_array)

In [ ]:
plot_adv_distances([results_rounded['adv_rounded'],adv], ['adv_rounded','adv'], combined_model, mal_x_batch, device)